# 1. Digital Twin Analyzer

# 🧠 **Project B — Telemetry Analyzer GUI**

### 📁 `analyzer/`  
Main folder for analysis and visualization.

## **1. GUI Layer (`gui/`)**
Handles all user interface components.

| File | Purpose |
|------|---------|
| `main_window.py` | Main GUI window, layout manager |
| `settings_panel.py` | File path, refresh interval, column selection |
| `module_panel.py` | Toggles for analysis modules |
| `progress_bar.py` | Shows file processing progress |
| `visualization_tabs.py` | Time series, clustering, forecasting, NLP, XAI |
| `log_panel.py` | Displays system logs and alerts |
| `health_summary.py` | Compact device status indicators |

---

## **2. Core Logic (`core/`)**
Handles file reading, analysis dispatch, and module execution.

| File | Purpose |
|------|---------|
| `analyzer_loop.py` | Main loop that polls file and dispatches modules |
| `reader.py` | Efficient tail reading of CSV/Parquet |
| `config_loader.py` | Loads shared `config.json` from generator |
| `alert_listener.py` | Listens to generator socket alerts |

---

## **3. Analysis Modules (`modules/`)**
Each module is optional and independently testable.

| File | Purpose |
|------|---------|
| `statistics.py` | Rolling means, variances, FFT |
| `clustering.py` | KMeans, DBSCAN, cluster visualization |
| `forecasting.py` | SARIMAX, PyTorch LSTM/GRU |
| `nlp.py` | spaCy, NLTK, topic modeling |
| `deep_learning.py` | Autoencoder, CNN, anomaly scoring |
| `xai.py` | SHAP, Captum, feature attribution |

---

## **4. Utilities (`utils/`)**

| File | Purpose |
|------|---------|
| `file_monitor.py` | Tracks file changes and row count |
| `plot_helpers.py` | Standardized plot generation |
| `alert_manager.py` | Formats and stores alert messages |

---

## **5. Entry Point**

| File | Purpose |
|------|---------|
| `app.py` | Launches the GUI and initializes components |

---

# 🧠 **Shared Config File (`config.json`)**
Written by Generator, read by Analyzer.

### Example contents:
```json
{
  "file_path": "telemetry.parquet",
  "columns": ["Temperature", "Motor RPM", "Error Code"],
  "sampling_rate_hz": 10,
  "timestamp_format": "ISO8601"
}
```

---

# 🧠 **Alert Socket (localhost:5050)**
- Generator sends JSON alerts:
  ```json
  { "event": "chunk_written", "timestamp": "2026-02-17T10:42:00", "rows": 10000 }
  ```
- Analyzer listens and refreshes immediately.

---

This structure is modular, scalable, and mirrors real-world telemetry systems. You can now implement each part independently, test in isolation, and extend with confidence.

# 2. GUI Folder

## 2.1. main_window.py

### `analyzer/gui/main_window.py`

```python
# analyzer/gui/main_window.py

from PySide6.QtWidgets import (
    QMainWindow,
    QWidget,
    QVBoxLayout,
    QHBoxLayout,
    QSplitter,
    QMessageBox,
)
from PySide6.QtCore import Qt

from .settings_panel import SettingsPanel
from .module_panel import ModulePanel
from .visualization_tabs import VisualizationTabs
from .log_panel import LogPanel
from .health_summary import HealthSummary
from .progress_bar import ProgressBar

from ..core.analyzer_loop import AnalyzerLoop
from ..core.config_loader import load_config
from ..core.alert_listener import AlertListener


class MainWindow(QMainWindow):
    """
    Main window for the Telemetry Analyzer GUI.

    Responsibilities:
        - Load shared config.json from Generator
        - Assemble all GUI panels
        - Start/stop the analyzer loop
        - Route alerts, logs, and visualization updates
    """

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Telemetry Data Analyzer")
        self.setMinimumSize(1400, 800)

        # Load shared configuration
        self.config = self._safe_load_config("config.json")

        # GUI panels
        self.settings_panel = SettingsPanel(self.config)
        self.module_panel = ModulePanel()
        self.visualization_tabs = VisualizationTabs()
        self.log_panel = LogPanel()
        self.health_summary = HealthSummary()
        self.progress_bar = ProgressBar()

        # Backend components
        self.analyzer: AnalyzerLoop | None = None
        self.alert_listener: AlertListener | None = None

        self._build_layout()
        self._connect_signals()
        self._init_backend()

    # ---------------------------------------------------------
    # Config loading
    # ---------------------------------------------------------
    def _safe_load_config(self, path: str):
        """
        Loads config.json and handles errors gracefully.
        """
        try:
            return load_config(path)
        except Exception as e:
            QMessageBox.warning(
                self,
                "Config Error",
                f"Could not load {path}.\n\n{e}\n\n"
                "You can still start the Analyzer, but some features may be disabled.",
            )
            return {}

    # ---------------------------------------------------------
    # Layout
    # ---------------------------------------------------------
    def _build_layout(self):
        central = QWidget()
        main_layout = QVBoxLayout()

        # --- Top area: left (settings + modules) / right (visualizations + health + log) ---
        splitter = QSplitter(Qt.Horizontal)

        # Left side: settings + module selection
        left_widget = QWidget()
        left_layout = QVBoxLayout()
        left_layout.addWidget(self.settings_panel)
        left_layout.addWidget(self.module_panel)
        left_layout.addStretch()
        left_widget.setLayout(left_layout)

        # Right side: visualizations + health summary + log
        right_widget = QWidget()
        right_layout = QVBoxLayout()
        right_layout.addWidget(self.visualization_tabs)
        right_layout.addWidget(self.health_summary)
        right_layout.addWidget(self.log_panel)
        right_widget.setLayout(right_layout)

        splitter.addWidget(left_widget)
        splitter.addWidget(right_widget)
        splitter.setStretchFactor(0, 1)
        splitter.setStretchFactor(1, 3)

        # Bottom: progress bar
        bottom_layout = QHBoxLayout()
        bottom_layout.addWidget(self.progress_bar)

        main_layout.addWidget(splitter)
        main_layout.addLayout(bottom_layout)

        central.setLayout(main_layout)
        self.setCentralWidget(central)

    # ---------------------------------------------------------
    # Signal wiring
    # ---------------------------------------------------------
    def _connect_signals(self):
        """
        Connects GUI signals to backend actions.
        """
        # Settings panel controls start/stop
        self.settings_panel.start_requested.connect(self._start_analysis)
        self.settings_panel.stop_requested.connect(self._stop_analysis)

        # Module selection changes which analyses run
        self.module_panel.modules_changed.connect(self._update_modules)

    # ---------------------------------------------------------
    # Backend initialization
    # ---------------------------------------------------------
    def _init_backend(self):
        """
        Initializes analyzer loop and alert listener.
        """
        # Analyzer loop: reads file, runs modules, updates GUI
        self.analyzer = AnalyzerLoop(
            config=self.config,
            progress_callback=self.progress_bar.update_progress,
            log_callback=self.log_panel.append_log,
            visualization_callback=self.visualization_tabs.update_visualizations,
            health_callback=self.health_summary.update_health,
        )

        # Alert listener: listens to Generator alerts on socket
        self.alert_listener = AlertListener(
            host=self.config.get("alerts", {}).get("socket_host", "127.0.0.1"),
            port=self.config.get("alerts", {}).get("socket_port", 5050),
            enabled=self.config.get("alerts", {}).get("socket_enabled", True),
            alert_callback=self._handle_alert,
        )
        self.alert_listener.start()

    # ---------------------------------------------------------
    # Analysis control
    # ---------------------------------------------------------
    def _start_analysis(self):
        """
        Starts the analyzer loop with current settings and selected modules.
        """
        if not self.analyzer:
            return

        analysis_config = self.settings_panel.get_analysis_config()
        selected_modules = self.module_panel.get_selected_modules()

        self.log_panel.append_log("Starting analysis...")
        self.analyzer.start(analysis_config, selected_modules)

    def _stop_analysis(self):
        """
        Stops the analyzer loop.
        """
        if not self.analyzer:
            return

        self.log_panel.append_log("Stopping analysis...")
        self.analyzer.stop()

    def _update_modules(self, modules):
        """
        Updates the analyzer with the currently selected modules.
        """
        if not self.analyzer:
            return
        self.analyzer.set_modules(modules)

    # ---------------------------------------------------------
    # Alert handling
    # ---------------------------------------------------------
    def _handle_alert(self, alert: dict):
        """
        Handles alerts received from the Generator via AlertListener.

        Example alert:
            {"event": "chunk_written", "payload": {"rows": 10000}}
        """
        event = alert.get("event", "unknown")
        payload = alert.get("payload", {})

        # Log the alert
        self.log_panel.append_log(f"[ALERT] {event} - {payload}")

        # Optional: trigger immediate refresh or health update
        if event == "chunk_written":
            self.health_summary.mark_data_updated()
        elif event == "generation_complete":
            self.health_summary.mark_generation_complete()

    # ---------------------------------------------------------
    # Cleanup
    # ---------------------------------------------------------
    def closeEvent(self, event):
        """
        Ensures backend threads are stopped cleanly on window close.
        """
        if self.analyzer:
            self.analyzer.stop()
        if self.alert_listener:
            self.alert_listener.stop()
        super().closeEvent(event)
```

---

## 🧠 Purpose

`MainWindow` is the **central orchestrator** of the Analyzer GUI. It:

- loads `config.json` written by the Generator  
- assembles all GUI components (settings, modules, visualizations, logs, health, progress)  
- initializes the backend (`AnalyzerLoop` and `AlertListener`)  
- routes signals between GUI and backend  
- ensures clean startup and shutdown  

It mirrors the role of Project A’s `MainWindow`, but for analysis instead of generation.

---

## 🧱 Structure

- **Config loading**
  - `_safe_load_config()` wraps `load_config("config.json")` and shows a warning if it fails.
- **Layout**
  - `QSplitter` horizontally:
    - Left: `SettingsPanel` + `ModulePanel`
    - Right: `VisualizationTabs` + `HealthSummary` + `LogPanel`
  - Bottom: `ProgressBar`
- **Backend**
  - `AnalyzerLoop` handles file reading and module execution.
  - `AlertListener` listens on the alert socket and forwards events to `_handle_alert`.

---

## 📥 Inputs

From other components:

- `config.json` via `load_config`
- User actions:
  - Start/stop analysis (buttons in `SettingsPanel`)
  - Module selection (`ModulePanel`)
- Alerts from Generator via `AlertListener`

Signals expected from GUI components:

- `SettingsPanel.start_requested`
- `SettingsPanel.stop_requested`
- `ModulePanel.modules_changed`

---

## 📤 Outputs

`MainWindow` coordinates:

- Calls to `AnalyzerLoop.start(config, modules)` and `.stop()`
- Updates to:
  - `ProgressBar.update_progress(percent, message)`
  - `LogPanel.append_log(text)`
  - `VisualizationTabs.update_visualizations(data)`
  - `HealthSummary.update_health(summary)`
- Reactions to alerts:
  - Logging alerts
  - Marking health status changes

No direct return values—everything is event/callback driven.

---

## 🔗 Integration Points

- **`AnalyzerLoop`** (to be implemented):
  - Must accept `config`, `progress_callback`, `log_callback`, `visualization_callback`, `health_callback`.
  - Must expose `start(analysis_config, selected_modules)`, `stop()`, and `set_modules(modules)`.
- **`AlertListener`** (to be implemented):
  - Listens on the same host/port as `AlertSocketClient` from Project A.
  - Calls `_handle_alert(alert_dict)` on each received message.
- **GUI panels** (to be implemented):
  - `SettingsPanel(config)` with:
    - `start_requested`, `stop_requested` signals
    - `get_analysis_config()` method
  - `ModulePanel()` with:
    - `modules_changed` signal
    - `get_selected_modules()` method
  - `VisualizationTabs.update_visualizations(data)`
  - `LogPanel.append_log(text)`
  - `HealthSummary.update_health(summary)`, `mark_data_updated()`, `mark_generation_complete()`
  - `ProgressBar.update_progress(percent, message)`

---


## 2.2. settings_panel.py

Below is the **complete, production‑ready implementation** of `analyzer/gui/settings_panel.py` — fully aligned with the architecture we defined for Project B.

This panel is the Analyzer’s **control cockpit**:  it lets the user configure *how* the Analyzer processes the telemetry file.

---

# 📄 `settings_panel.py`

```python
# analyzer/gui/settings_panel.py

from PySide6.QtWidgets import (
    QWidget,
    QGroupBox,
    QVBoxLayout,
    QHBoxLayout,
    QLabel,
    QSpinBox,
    QLineEdit,
    QPushButton,
    QFileDialog,
)
from PySide6.QtCore import Qt, Signal


class SettingsPanel(QWidget):
    """
    Settings panel for the Telemetry Analyzer.

    Responsibilities:
        - Display file path (from config.json or user override)
        - Allow user to change refresh interval (polling frequency)
        - Allow user to limit number of rows loaded per refresh
        - Provide Start / Stop buttons for analysis

    Signals:
        start_requested -> emitted when user presses Start
        stop_requested  -> emitted when user presses Stop

    Methods:
        get_analysis_config() -> dict
            Returns current analysis settings for AnalyzerLoop
    """

    start_requested = Signal()
    stop_requested = Signal()

    def __init__(self, config: dict):
        super().__init__()

        self.config = config or {}

        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        # --- File Path ---
        file_group = QGroupBox("Telemetry File")
        file_layout = QVBoxLayout()

        self.file_path_edit = QLineEdit()
        self.file_path_edit.setText(self.config.get("output", {}).get("file_path", ""))

        browse_btn = QPushButton("Browse…")
        browse_btn.clicked.connect(self._browse_file)

        file_layout.addWidget(QLabel("File path:"))
        file_layout.addWidget(self.file_path_edit)
        file_layout.addWidget(browse_btn)
        file_group.setLayout(file_layout)

        # --- Refresh Interval ---
        refresh_group = QGroupBox("Refresh Settings")
        refresh_layout = QVBoxLayout()

        self.refresh_spin = QSpinBox()
        self.refresh_spin.setRange(100, 10_000)
        self.refresh_spin.setValue(500)  # ms
        self.refresh_spin.setSuffix(" ms")

        refresh_layout.addWidget(QLabel("Refresh interval:"))
        refresh_layout.addWidget(self.refresh_spin)
        refresh_group.setLayout(refresh_layout)

        # --- Row Limit ---
        row_group = QGroupBox("Row Processing")
        row_layout = QVBoxLayout()

        self.row_limit_spin = QSpinBox()
        self.row_limit_spin.setRange(100, 1_000_000)
        self.row_limit_spin.setValue(10_000)

        row_layout.addWidget(QLabel("Rows per refresh:"))
        row_layout.addWidget(self.row_limit_spin)
        row_group.setLayout(row_layout)

        # --- Start / Stop Buttons ---
        btn_layout = QHBoxLayout()
        self.start_btn = QPushButton("Start Analysis")
        self.stop_btn = QPushButton("Stop")

        self.start_btn.clicked.connect(self.start_requested.emit)
        self.stop_btn.clicked.connect(self.stop_requested.emit)

        btn_layout.addWidget(self.start_btn)
        btn_layout.addWidget(self.stop_btn)

        # Assemble layout
        layout.addWidget(file_group)
        layout.addWidget(refresh_group)
        layout.addWidget(row_group)
        layout.addLayout(btn_layout)
        layout.addStretch()

        self.setLayout(layout)

    # ---------------------------------------------------------
    # File Browser
    # ---------------------------------------------------------
    def _browse_file(self):
        """
        Opens a file dialog to select a CSV or Parquet file.
        """
        path, _ = QFileDialog.getOpenFileName(
            self,
            "Select Telemetry File",
            "",
            "Data Files (*.csv *.parquet)"
        )
        if path:
            self.file_path_edit.setText(path)

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def get_analysis_config(self) -> dict:
        """
        Returns a dictionary with current analysis settings.
        Used by AnalyzerLoop.start().
        """
        return {
            "file_path": self.file_path_edit.text(),
            "refresh_interval_ms": self.refresh_spin.value(),
            "row_limit": self.row_limit_spin.value(),
        }
```

---

# 🧠 **Purpose**

`SettingsPanel` is the Analyzer’s **control center**.  
It lets the user configure:

- **Which file** to analyze (default from `config.json`, but user can override)
- **How often** the Analyzer should refresh (polling interval)
- **How many rows** to process per refresh (batch size)
- **Start/Stop** of the analysis loop

It is the equivalent of Project A’s `SettingsPanel`, but for *analysis* instead of *generation*.

---

# 🧱 **Structure**

### 1. **Telemetry File Section**
- Shows the file path from `config.json`
- Allows browsing for a different file

### 2. **Refresh Settings**
- Polling interval in milliseconds  
- Controls how often `AnalyzerLoop` reads new data

### 3. **Row Processing**
- Maximum number of rows to load per refresh  
- Prevents huge memory spikes

### 4. **Start / Stop Buttons**
- Emit Qt signals:
  - `start_requested`
  - `stop_requested`

These are connected in `MainWindow` to:

```python
self.settings_panel.start_requested.connect(self._start_analysis)
self.settings_panel.stop_requested.connect(self._stop_analysis)
```

---

# 📥 **Inputs**

- `config` dictionary loaded from `config.json`
  - Used to pre-fill the file path

- User interactions:
  - File path editing
  - Refresh interval selection
  - Row limit selection
  - Start/Stop buttons

---

# 📤 **Outputs**

### 1. **Signals**
- `start_requested`
- `stop_requested`

### 2. **Analysis configuration dictionary**
Returned by:

```python
settings_panel.get_analysis_config()
```

Example:

```python
{
  "file_path": "telemetry.parquet",
  "refresh_interval_ms": 500,
  "row_limit": 10000
}
```

This is passed directly to:

```python
AnalyzerLoop.start(analysis_config, selected_modules)
```

---

# 🔗 **Integration with the Analyzer**

`MainWindow` uses this panel to:

- Start the analyzer loop  
- Stop the analyzer loop  
- Provide analysis settings to `AnalyzerLoop`  

The AnalyzerLoop then:

- Reads the file  
- Processes new rows  
- Dispatches modules  
- Updates visualizations  
- Updates health summary  
- Updates progress bar  

---


## 2.3. module_panel.py

Here is the **complete, production‑ready implementation** of `analyzer/gui/module_panel.py`, written in the same architectural style as the rest of Project B.  

This panel is the Analyzer’s **module selector** — the user chooses which analysis modules to run.

---

# 📄 `module_panel.py`

```python
# analyzer/gui/module_panel.py

from PySide6.QtWidgets import (
    QWidget,
    QGroupBox,
    QVBoxLayout,
    QCheckBox,
)
from PySide6.QtCore import Signal


class ModulePanel(QWidget):
    """
    Module selection panel for the Telemetry Analyzer.

    Responsibilities:
        - Display toggles for each analysis module
        - Allow user to enable/disable modules dynamically
        - Emit a signal when module selection changes

    Signals:
        modules_changed -> emitted with a list of selected module names

    Methods:
        get_selected_modules() -> list[str]
            Returns the list of currently enabled modules
    """

    modules_changed = Signal(list)

    def __init__(self):
        super().__init__()
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        group = QGroupBox("Analysis Modules")
        group_layout = QVBoxLayout()

        # Define available modules
        self.module_checkboxes = {
            "statistics": QCheckBox("Statistics (rolling means, FFT)"),
            "clustering": QCheckBox("Clustering (KMeans, DBSCAN)"),
            "forecasting": QCheckBox("Forecasting (SARIMAX, LSTM)"),
            "nlp": QCheckBox("NLP (logs, topic modeling)"),
            "deep_learning": QCheckBox("Deep Learning (autoencoder anomaly detection)"),
            "xai": QCheckBox("Explainability (SHAP, Captum)"),
        }

        # Add checkboxes to layout
        for name, checkbox in self.module_checkboxes.items():
            checkbox.stateChanged.connect(self._emit_change)
            group_layout.addWidget(checkbox)

        group.setLayout(group_layout)
        layout.addWidget(group)
        layout.addStretch()

        self.setLayout(layout)

    # ---------------------------------------------------------
    # Signal emission
    # ---------------------------------------------------------
    def _emit_change(self):
        """
        Emits the modules_changed signal whenever a checkbox is toggled.
        """
        self.modules_changed.emit(self.get_selected_modules())

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def get_selected_modules(self) -> list:
        """
        Returns a list of module names that are currently enabled.
        """
        return [
            name
            for name, checkbox in self.module_checkboxes.items()
            if checkbox.isChecked()
        ]
```

---

# 🧠 **Purpose**

`ModulePanel` is the Analyzer’s **module selector**.  
It allows the user to choose which analysis modules should run during each refresh cycle.

This includes:

- **Statistics** (rolling means, FFT, correlations)  
- **Clustering** (KMeans, DBSCAN)  
- **Forecasting** (SARIMAX, LSTM/GRU)  
- **NLP** (log analysis, topic modeling)  
- **Deep Learning** (autoencoder anomaly detection)  
- **XAI** (SHAP, Captum feature attribution)  

Each module is optional and independently testable.

---

# 🧱 **Structure**

### 1. A `QGroupBox` titled “Analysis Modules”
Inside it, six checkboxes — one for each module.

### 2. A dictionary mapping module names → checkboxes
```python
self.module_checkboxes = {
    "statistics": QCheckBox(...),
    "clustering": QCheckBox(...),
    ...
}
```

### 3. A signal `modules_changed`
Emitted whenever the user toggles a module.

### 4. A public method `get_selected_modules()`
Returns a list of enabled module names.

---

# 📥 **Inputs**

- User toggles checkboxes  
- No external inputs required  

---

# 📤 **Outputs**

### 1. **Signal: `modules_changed`**
Emitted with a list of selected modules:

```python
["statistics", "forecasting", "xai"]
```

### 2. **Method: `get_selected_modules()`**
Used by `MainWindow` when starting analysis:

```python
selected_modules = self.module_panel.get_selected_modules()
```

---

# 🔗 **Integration with the Analyzer**

### In `MainWindow._connect_signals()`:

```python
self.module_panel.modules_changed.connect(self._update_modules)
```

### In `MainWindow._update_modules()`:

```python
self.analyzer.set_modules(modules)
```

### In `AnalyzerLoop`:

- Only the selected modules are executed each refresh cycle  
- Modules are dynamically hot‑swappable during runtime  

This makes the Analyzer extremely flexible.

---


## 2.4. progress_bar.py

Here is the **complete, production‑ready implementation** of `analyzer/gui/progress_bar.py`, written in the same clean, modular style as the rest of Project B.

This component mirrors the Generator’s status bar but is tailored for **file processing progress** during analysis.

---

# 📄 `progress_bar.py`

```python
# analyzer/gui/progress_bar.py

from PySide6.QtWidgets import QWidget, QHBoxLayout, QLabel, QProgressBar
from PySide6.QtCore import Qt


class ProgressBar(QWidget):
    """
    Progress bar for the Telemetry Analyzer.

    Responsibilities:
        - Display progress of file processing
        - Show status messages (e.g., "Reading...", "Analyzing...")
        - Provide a clean, compact UI element for the bottom of MainWindow

    Methods:
        update_progress(percent, message)
            Updates the progress bar and status label
    """

    def __init__(self):
        super().__init__()
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QHBoxLayout()

        # Progress bar
        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)
        self.progress.setFormat("0%")
        self.progress.setTextVisible(True)

        # Status message
        self.message_label = QLabel("Idle.")
        self.message_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)

        layout.addWidget(self.progress, stretch=2)
        layout.addWidget(self.message_label, stretch=3)

        self.setLayout(layout)

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def update_progress(self, percent: float, message: str = ""):
        """
        Updates the progress bar and optional status message.

        Args:
            percent (float): 0–100 progress value
            message (str): Optional status message
        """
        self.progress.setValue(int(percent))
        if message:
            self.message_label.setText(message)
```

---

# 🧠 Purpose

The Analyzer’s `ProgressBar` is a **compact status indicator** that shows:

- how much of the telemetry file has been processed  
- what the Analyzer is currently doing  
  - “Reading new rows…”  
  - “Running statistics…”  
  - “Clustering…”  
  - “Forecasting…”  

It sits at the bottom of the Analyzer’s main window and updates continuously during analysis.

---

# 🧱 Structure

### 1. **Progress Bar**
- Shows 0–100% progress  
- Updated by `AnalyzerLoop` via callback  

### 2. **Status Message Label**
- Displays human‑readable status text  
- Updated together with progress  

### 3. **Public Method: `update_progress()`**
Called by the backend:

```python
progress_callback(percent, message)
```

This is passed into `AnalyzerLoop` during initialization.

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
self.progress_callback(42.0, "Analyzing new rows…")
```

Inputs include:

- `percent`: float (0–100)  
- `message`: optional string  

---

# 📤 Outputs

The widget updates:

- the progress bar  
- the status label  

No return values — purely visual.

---

# 🔗 Integration with the Analyzer

### In `MainWindow.__init__`:

```python
self.progress_bar = ProgressBar()
```

### In `_init_backend()`:

```python
self.analyzer = AnalyzerLoop(
    config=self.config,
    progress_callback=self.progress_bar.update_progress,
    ...
)
```

### In `AnalyzerLoop`:

```python
self.progress_callback(percent, "Reading new data…")
```

This keeps the user informed about the Analyzer’s progress in real time.

---


## 2.5. visualization_tabs.py

Here is the **complete, production‑ready implementation** of `analyzer/gui/visualization_tabs.py` — one of the most important GUI components in Project B.

This module provides a **tabbed visualization interface** where each analysis module can render its results:

- Time‑series plots  
- Clustering scatterplots  
- Forecasting curves  
- NLP summaries  
- Deep learning anomaly scores  
- XAI feature attributions  

It is intentionally modular: each tab exposes a simple `update_*()` method so the backend (`AnalyzerLoop`) can push new data without knowing anything about the GUI.

---

# 📄 `visualization_tabs.py`

```python
# analyzer/gui/visualization_tabs.py

from PySide6.QtWidgets import (
    QWidget,
    QTabWidget,
    QVBoxLayout,
    QLabel,
)
from PySide6.QtCore import Qt

from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class VisualizationTabs(QWidget):
    """
    Tabbed visualization interface for the Telemetry Analyzer.

    Responsibilities:
        - Provide separate tabs for each analysis module
        - Expose update_*() methods for AnalyzerLoop to push results
        - Render plots using Matplotlib
        - Keep GUI decoupled from analysis logic

    Tabs:
        - Time Series
        - Clustering
        - Forecasting
        - NLP
        - Deep Learning
        - XAI
    """

    def __init__(self):
        super().__init__()
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        self.tabs = QTabWidget()

        # --- Time Series Tab ---
        self.ts_fig = Figure(figsize=(5, 3))
        self.ts_canvas = FigureCanvas(self.ts_fig)
        self.ts_ax = self.ts_fig.add_subplot(111)
        ts_widget = QWidget()
        ts_layout = QVBoxLayout()
        ts_layout.addWidget(self.ts_canvas)
        ts_widget.setLayout(ts_layout)
        self.tabs.addTab(ts_widget, "Time Series")

        # --- Clustering Tab ---
        self.cluster_fig = Figure(figsize=(5, 3))
        self.cluster_canvas = FigureCanvas(self.cluster_fig)
        self.cluster_ax = self.cluster_fig.add_subplot(111)
        cluster_widget = QWidget()
        cluster_layout = QVBoxLayout()
        cluster_layout.addWidget(self.cluster_canvas)
        cluster_widget.setLayout(cluster_layout)
        self.tabs.addTab(cluster_widget, "Clustering")

        # --- Forecasting Tab ---
        self.forecast_fig = Figure(figsize=(5, 3))
        self.forecast_canvas = FigureCanvas(self.forecast_fig)
        self.forecast_ax = self.forecast_fig.add_subplot(111)
        forecast_widget = QWidget()
        forecast_layout = QVBoxLayout()
        forecast_layout.addWidget(self.forecast_canvas)
        forecast_widget.setLayout(forecast_layout)
        self.tabs.addTab(forecast_widget, "Forecasting")

        # --- NLP Tab ---
        self.nlp_label = QLabel("NLP results will appear here.")
        self.nlp_label.setAlignment(Qt.AlignTop | Qt.AlignLeft)
        nlp_widget = QWidget()
        nlp_layout = QVBoxLayout()
        nlp_layout.addWidget(self.nlp_label)
        nlp_widget.setLayout(nlp_layout)
        self.tabs.addTab(nlp_widget, "NLP")

        # --- Deep Learning Tab ---
        self.dl_fig = Figure(figsize=(5, 3))
        self.dl_canvas = FigureCanvas(self.dl_fig)
        self.dl_ax = self.dl_fig.add_subplot(111)
        dl_widget = QWidget()
        dl_layout = QVBoxLayout()
        dl_layout.addWidget(self.dl_canvas)
        dl_widget.setLayout(dl_layout)
        self.tabs.addTab(dl_widget, "Deep Learning")

        # --- XAI Tab ---
        self.xai_label = QLabel("XAI feature attributions will appear here.")
        self.xai_label.setAlignment(Qt.AlignTop | Qt.AlignLeft)
        xai_widget = QWidget()
        xai_layout = QVBoxLayout()
        xai_layout.addWidget(self.xai_label)
        xai_widget.setLayout(xai_layout)
        self.tabs.addTab(xai_widget, "XAI")

        layout.addWidget(self.tabs)
        self.setLayout(layout)

    # ---------------------------------------------------------
    # Update Methods (called by AnalyzerLoop)
    # ---------------------------------------------------------
    def update_visualizations(self, results: dict):
        """
        Central update entry point.
        AnalyzerLoop sends a dict with keys:
            - "time_series"
            - "clustering"
            - "forecasting"
            - "nlp"
            - "deep_learning"
            - "xai"
        Each key maps to module-specific results.
        """
        if "time_series" in results:
            self.update_time_series(results["time_series"])

        if "clustering" in results:
            self.update_clustering(results["clustering"])

        if "forecasting" in results:
            self.update_forecasting(results["forecasting"])

        if "nlp" in results:
            self.update_nlp(results["nlp"])

        if "deep_learning" in results:
            self.update_deep_learning(results["deep_learning"])

        if "xai" in results:
            self.update_xai(results["xai"])

    # ---------------------------------------------------------
    # Individual Update Methods
    # ---------------------------------------------------------
    def update_time_series(self, data):
        """
        Expects:
            data = {
                "x": [...],
                "y": [...],
                "label": "Temperature"
            }
        """
        self.ts_ax.clear()
        self.ts_ax.plot(data["x"], data["y"], label=data.get("label", ""))
        self.ts_ax.set_title("Time Series")
        self.ts_ax.legend()
        self.ts_canvas.draw_idle()

    def update_clustering(self, data):
        """
        Expects:
            data = {
                "x": [...],
                "y": [...],
                "labels": [...]
            }
        """
        self.cluster_ax.clear()
        self.cluster_ax.scatter(data["x"], data["y"], c=data["labels"], cmap="viridis")
        self.cluster_ax.set_title("Clustering")
        self.cluster_canvas.draw_idle()

    def update_forecasting(self, data):
        """
        Expects:
            data = {
                "history_x": [...],
                "history_y": [...],
                "forecast_x": [...],
                "forecast_y": [...]
            }
        """
        self.forecast_ax.clear()
        self.forecast_ax.plot(data["history_x"], data["history_y"], label="History")
        self.forecast_ax.plot(data["forecast_x"], data["forecast_y"], label="Forecast")
        self.forecast_ax.set_title("Forecasting")
        self.forecast_ax.legend()
        self.forecast_canvas.draw_idle()

    def update_nlp(self, text: str):
        """
        Expects:
            text = "Summary of log messages..."
        """
        self.nlp_label.setText(text)

    def update_deep_learning(self, data):
        """
        Expects:
            data = {
                "x": [...],
                "anomaly_score": [...]
            }
        """
        self.dl_ax.clear()
        self.dl_ax.plot(data["x"], data["anomaly_score"], color="red")
        self.dl_ax.set_title("Anomaly Score")
        self.dl_canvas.draw_idle()

    def update_xai(self, text: str):
        """
        Expects:
            text = "Feature attribution summary..."
        """
        self.xai_label.setText(text)
```

---

# 🧠 Purpose

`VisualizationTabs` is the Analyzer’s **visual output hub**.  
It provides a clean, modular interface for rendering:

- time‑series plots  
- clustering scatterplots  
- forecasting curves  
- NLP summaries  
- deep learning anomaly scores  
- XAI feature attributions  

It is the GUI counterpart to the analysis modules in `analyzer/modules/`.

---

# 🧱 Structure

### 1. **QTabWidget**
Contains six tabs:

| Tab | Purpose |
|-----|---------|
| Time Series | Rolling sensor plots |
| Clustering | KMeans/DBSCAN scatterplots |
| Forecasting | SARIMAX/LSTM predictions |
| NLP | Log summaries, topics |
| Deep Learning | Autoencoder anomaly scores |
| XAI | SHAP/Captum explanations |

### 2. **Matplotlib Figures**
Three tabs use Matplotlib:

- Time Series  
- Clustering  
- Forecasting  
- Deep Learning  

### 3. **Text Labels**
Two tabs use simple text:

- NLP  
- XAI  

### 4. **Unified Update Entry Point**
`update_visualizations(results: dict)`  
The backend sends a dictionary with module outputs.

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
visualization_callback(results_dict)
```

Where `results_dict` may contain:

```python
{
  "time_series": {...},
  "clustering": {...},
  "forecasting": {...},
  "nlp": "Summary text",
  "deep_learning": {...},
  "xai": "Attribution text"
}
```

---

# 📤 Outputs

- Updated Matplotlib plots  
- Updated text labels  
- No return values — purely visual  

---

# 🔗 Integration with the Analyzer

### In `MainWindow._init_backend()`:

```python
self.analyzer = AnalyzerLoop(
    config=self.config,
    progress_callback=self.progress_bar.update_progress,
    log_callback=self.log_panel.append_log,
    visualization_callback=self.visualization_tabs.update_visualizations,
    health_callback=self.health_summary.update_health,
)
```

### In `AnalyzerLoop`:

Each module returns its results, and the loop aggregates them:

```python
results = {}
results["time_series"] = ts_output
results["clustering"] = cluster_output
...
self.visualization_callback(results)
```

---


## 2.6. log_panel.py

Here is the **complete, production‑ready implementation** of `analyzer/gui/log_panel.py`, written in the same clean, modular style as the rest of Project B.

This panel is the Analyzer’s **live event console**.  
It displays:

- system logs  
- module outputs  
- alerts from the Generator  
- internal warnings  
- analysis status messages  

It is intentionally simple, fast, and append‑only — perfect for real‑time telemetry analysis.  

---

# 📄 `log_panel.py`

```python
# analyzer/gui/log_panel.py

from PySide6.QtWidgets import QWidget, QGroupBox, QVBoxLayout, QTextEdit
from PySide6.QtCore import Qt
from datetime import datetime


class LogPanel(QWidget):
    """
    Log panel for the Telemetry Analyzer.

    Responsibilities:
        - Display system logs, alerts, and module messages
        - Provide an append-only text console
        - Timestamp each entry for clarity

    Methods:
        append_log(text: str)
            Appends a timestamped log entry to the console
    """

    def __init__(self):
        super().__init__()
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        group = QGroupBox("System Log")
        group_layout = QVBoxLayout()

        self.text_area = QTextEdit()
        self.text_area.setReadOnly(True)
        self.text_area.setLineWrapMode(QTextEdit.NoWrap)

        group_layout.addWidget(self.text_area)
        group.setLayout(group_layout)

        layout.addWidget(group)
        self.setLayout(layout)

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def append_log(self, text: str):
        """
        Appends a timestamped log entry to the console.

        Args:
            text (str): Log message
        """
        timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        entry = f"[{timestamp}] {text}"

        self.text_area.append(entry)
        self.text_area.ensureCursorVisible()
```

---

# 🧠 Purpose

`LogPanel` is the Analyzer’s **real‑time console**.  
It provides a clear, timestamped stream of:

- backend events  
- module outputs  
- alerts from the Generator  
- warnings and errors  
- analysis progress messages  

It is essential for debugging, monitoring, and understanding system behavior.

---

# 🧱 Structure

### 1. **QGroupBox: “System Log”**
A clean container for the log console.

### 2. **QTextEdit (read‑only)**
- Append‑only  
- No line wrapping (better for structured logs)  
- Auto‑scrolls to the newest entry  

### 3. **Public Method: `append_log(text)`**
Adds a timestamped entry:

```
[2026-02-19 10:42:00] Starting analysis...
```

---

# 📥 Inputs

From:

- `MainWindow` (alerts, start/stop messages)
- `AnalyzerLoop` (module outputs, progress messages)
- `AlertListener` (Generator alerts)

Example:

```python
self.log_panel.append_log("New rows detected in file.")
```

---

# 📤 Outputs

- Updated log console  
- No return values — purely visual  

---

# 🔗 Integration with the Analyzer

### In `MainWindow._init_backend()`:

```python
self.analyzer = AnalyzerLoop(
    config=self.config,
    progress_callback=self.progress_bar.update_progress,
    log_callback=self.log_panel.append_log,
    visualization_callback=self.visualization_tabs.update_visualizations,
    health_callback=self.health_summary.update_health,
)
```

### In `AnalyzerLoop`:

```python
self.log_callback("Running statistics module...")
```

### In `AlertListener`:

```python
self.log_panel.append_log(f"[ALERT] {event} - {payload}")
```

---


## 2.7. health_summary.py

Below is the **complete, production‑ready implementation** of `analyzer/gui/health_summary.py`, written in the same clean, modular style as the rest of Project B.

This panel provides a **compact, color‑coded health indicator** for the telemetry system.  
It reacts to:

- new data arriving  
- generator alerts  
- analysis results  
- anomalies detected by modules  

It is intentionally lightweight and visually expressive.

---

# 📄 `health_summary.py`

```python
# analyzer/gui/health_summary.py

from PySide6.QtWidgets import QWidget, QGroupBox, QVBoxLayout, QLabel, QHBoxLayout
from PySide6.QtCore import Qt
from datetime import datetime


class HealthSummary(QWidget):
    """
    Compact health indicator panel for the Telemetry Analyzer.

    Responsibilities:
        - Display system health status (OK / Warning / Error)
        - Show last update time
        - React to alerts from the Generator
        - React to analysis results (e.g., anomalies)

    Methods:
        update_health(summary: dict)
            Called by AnalyzerLoop with module-derived health info

        mark_data_updated()
            Called when new data arrives (e.g., chunk_written alert)

        mark_generation_complete()
            Called when generator finishes producing data
    """

    def __init__(self):
        super().__init__()
        self._build_ui()

        # Internal state
        self.last_update = None
        self.status = "OK"

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        group = QGroupBox("System Health")
        group_layout = QVBoxLayout()

        # Status label (color-coded)
        self.status_label = QLabel("Status: OK")
        self.status_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)
        self._set_status_color("OK")

        # Last update timestamp
        self.update_label = QLabel("Last update: —")
        self.update_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)

        # Generator state
        self.generator_label = QLabel("Generator: Active")
        self.generator_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)

        group_layout.addWidget(self.status_label)
        group_layout.addWidget(self.update_label)
        group_layout.addWidget(self.generator_label)

        group.setLayout(group_layout)
        layout.addWidget(group)
        self.setLayout(layout)

    # ---------------------------------------------------------
    # Status Color Helper
    # ---------------------------------------------------------
    def _set_status_color(self, status: str):
        """
        Applies color coding based on status.
        """
        if status == "OK":
            color = "green"
        elif status == "Warning":
            color = "orange"
        else:
            color = "red"

        self.status_label.setStyleSheet(f"color: {color}; font-weight: bold;")

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def update_health(self, summary: dict):
        """
        Called by AnalyzerLoop with health information.

        Expected summary format:
            {
                "status": "OK" | "Warning" | "Error",
                "message": "Optional description"
            }
        """
        status = summary.get("status", "OK")
        message = summary.get("message", "")

        self.status = status
        self._set_status_color(status)

        if message:
            self.status_label.setText(f"Status: {status} — {message}")
        else:
            self.status_label.setText(f"Status: {status}")

        # Update timestamp
        self.last_update = datetime.utcnow()
        self.update_label.setText(f"Last update: {self.last_update.strftime('%H:%M:%S')}")

    def mark_data_updated(self):
        """
        Called when new data arrives (e.g., chunk_written alert).
        """
        self.last_update = datetime.utcnow()
        self.update_label.setText(f"Last update: {self.last_update.strftime('%H:%M:%S')}")

    def mark_generation_complete(self):
        """
        Called when generator finishes producing data.
        """
        self.generator_label.setText("Generator: Complete")
        self.generator_label.setStyleSheet("color: blue; font-weight: bold;")
```

---

# 🧠 Purpose

`HealthSummary` is the Analyzer’s **system health dashboard**.  
It gives the user a quick, at‑a‑glance understanding of:

- whether the system is behaving normally  
- whether anomalies or warnings were detected  
- when the last data update occurred  
- whether the Generator is still producing data  

This mirrors real industrial telemetry dashboards.

---

# 🧱 Structure

### 1. **Status Label**
Color‑coded:

- Green → OK  
- Orange → Warning  
- Red → Error  

### 2. **Last Update Timestamp**
Shows when the Analyzer last received:

- new data  
- new analysis results  
- new alerts  

### 3. **Generator State**
Shows:

- “Active”  
- “Complete” (after receiving `generation_complete` alert)  

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
health_callback({"status": "Warning", "message": "High vibration detected"})
```

From `MainWindow` (alert listener):

```python
self.health_summary.mark_data_updated()
```

From `MainWindow` (generator complete):

```python
self.health_summary.mark_generation_complete()
```

---

# 📤 Outputs

- Updated health status  
- Updated timestamp  
- Updated generator state  

No return values — purely visual.

---

# 🔗 Integration with the Analyzer

### In `MainWindow._init_backend()`:

```python
self.analyzer = AnalyzerLoop(
    config=self.config,
    progress_callback=self.progress_bar.update_progress,
    log_callback=self.log_panel.append_log,
    visualization_callback=self.visualization_tabs.update_visualizations,
    health_callback=self.health_summary.update_health,
)
```

### In `AnalyzerLoop`:

Each module can contribute to health:

```python
health = {"status": "Warning", "message": "RPM variance high"}
self.health_callback(health)
```

### In `AlertListener`:

```python
if event == "chunk_written":
    self.health_summary.mark_data_updated()
elif event == "generation_complete":
    self.health_summary.mark_generation_complete()
```

---


# 3. Core Folder

## 3.1. analyzer_loop.py

### `analyzer/core/analyzer_loop.py`

```python
# analyzer/core/analyzer_loop.py

import threading
import time
from typing import Callable, Dict, List, Any, Optional

from .reader import TelemetryReader
from ..modules import statistics, clustering, forecasting, nlp, deep_learning, xai


class AnalyzerLoop:
    """
    Main analysis loop for the Telemetry Analyzer.

    Responsibilities:
        - Periodically read new data from the telemetry file
        - Dispatch selected analysis modules
        - Aggregate results and send them to the GUI
        - Report progress, logs, and health status

    Lifecycle:
        - start(analysis_config, selected_modules)
        - loop in background thread until stop() is called
    """

    def __init__(
        self,
        config: dict,
        progress_callback: Callable[[float, str], None],
        log_callback: Callable[[str], None],
        visualization_callback: Callable[[Dict[str, Any]], None],
        health_callback: Callable[[Dict[str, Any]], None],
    ):
        self.config = config or {}
        self.progress_callback = progress_callback
        self.log_callback = log_callback
        self.visualization_callback = visualization_callback
        self.health_callback = health_callback

        self.reader: Optional[TelemetryReader] = None
        self.selected_modules: List[str] = []

        self._thread: Optional[threading.Thread] = None
        self._stop_event = threading.Event()

        # Internal state
        self.total_rows_processed = 0

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def start(self, analysis_config: dict, selected_modules: List[str]):
        """
        Starts the analyzer loop in a background thread.

        analysis_config:
            {
                "file_path": str,
                "refresh_interval_ms": int,
                "row_limit": int
            }

        selected_modules:
            ["statistics", "clustering", ...]
        """
        if self._thread and self._thread.is_alive():
            self.log_callback("AnalyzerLoop is already running.")
            return

        file_path = analysis_config.get("file_path")
        if not file_path:
            self.log_callback("No file path specified for analysis.")
            return

        self.selected_modules = selected_modules
        self.reader = TelemetryReader(
            file_path=file_path,
            row_limit=analysis_config.get("row_limit", 10_000),
        )

        self.refresh_interval = analysis_config.get("refresh_interval_ms", 500) / 1000.0

        self.log_callback(f"Starting analysis on file: {file_path}")
        self.total_rows_processed = 0
        self._stop_event.clear()

        self._thread = threading.Thread(target=self._run_loop, daemon=True)
        self._thread.start()

    def stop(self):
        """
        Requests the analyzer loop to stop and waits for thread to finish.
        """
        if not self._thread:
            return

        self._stop_event.set()
        self._thread.join(timeout=2.0)
        self.log_callback("AnalyzerLoop stopped.")

    def set_modules(self, modules: List[str]):
        """
        Updates the list of selected modules at runtime.
        """
        self.selected_modules = modules
        self.log_callback(f"Modules updated: {modules}")

    # ---------------------------------------------------------
    # Internal Loop
    # ---------------------------------------------------------
    def _run_loop(self):
        """
        Background loop:
            - periodically reads new data
            - runs selected modules
            - updates GUI via callbacks
        """
        while not self._stop_event.is_set():
            try:
                # 1. Read new data
                new_data = self.reader.read_new_rows() if self.reader else None
                if new_data is None or new_data.empty:
                    # No new data; sleep and continue
                    self.progress_callback(0.0, "Waiting for new data...")
                    time.sleep(self.refresh_interval)
                    continue

                rows = len(new_data)
                self.total_rows_processed += rows
                self.log_callback(f"Read {rows} new rows (total: {self.total_rows_processed}).")

                # 2. Run selected modules
                results: Dict[str, Any] = {}
                health_updates: List[Dict[str, Any]] = []

                if "statistics" in self.selected_modules:
                    stats_result, stats_health = statistics.run(new_data)
                    results["time_series"] = stats_result
                    if stats_health:
                        health_updates.append(stats_health)

                if "clustering" in self.selected_modules:
                    cluster_result, cluster_health = clustering.run(new_data)
                    results["clustering"] = cluster_result
                    if cluster_health:
                        health_updates.append(cluster_health)

                if "forecasting" in self.selected_modules:
                    forecast_result, forecast_health = forecasting.run(new_data)
                    results["forecasting"] = forecast_result
                    if forecast_health:
                        health_updates.append(forecast_health)

                if "nlp" in self.selected_modules:
                    nlp_result, nlp_health = nlp.run(new_data)
                    results["nlp"] = nlp_result
                    if nlp_health:
                        health_updates.append(nlp_health)

                if "deep_learning" in self.selected_modules:
                    dl_result, dl_health = deep_learning.run(new_data)
                    results["deep_learning"] = dl_result
                    if dl_health:
                        health_updates.append(dl_health)

                if "xai" in self.selected_modules:
                    xai_result, xai_health = xai.run(new_data)
                    results["xai"] = xai_result
                    if xai_health:
                        health_updates.append(xai_health)

                # 3. Push visualization updates
                if results:
                    self.visualization_callback(results)

                # 4. Aggregate health
                if health_updates:
                    # Simple aggregation: worst status wins
                    aggregated = self._aggregate_health(health_updates)
                    self.health_callback(aggregated)

                # 5. Update progress (coarse: we don't know total rows, so use activity)
                self.progress_callback(0.0, f"Processed {self.total_rows_processed} rows.")

                # 6. Sleep until next refresh
                time.sleep(self.refresh_interval)

            except Exception as e:
                self.log_callback(f"Error in AnalyzerLoop: {e}")
                self.health_callback({"status": "Error", "message": str(e)})
                time.sleep(self.refresh_interval)

    # ---------------------------------------------------------
    # Health Aggregation
    # ---------------------------------------------------------
    def _aggregate_health(self, health_list: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Aggregates multiple health dicts into a single summary.

        Priority: Error > Warning > OK
        """
        priority = {"OK": 0, "Warning": 1, "Error": 2}
        best_status = "OK"
        messages = []

        for h in health_list:
            status = h.get("status", "OK")
            msg = h.get("message", "")
            if msg:
                messages.append(msg)
            if priority.get(status, 0) > priority.get(best_status, 0):
                best_status = status

        return {
            "status": best_status,
            "message": " | ".join(messages) if messages else "",
        }
```

---

### Purpose

`AnalyzerLoop` is the **heart of Project B**. It:

- runs in a background thread  
- periodically reads new telemetry rows from the file  
- dispatches the selected analysis modules  
- aggregates their outputs into:
  - visualization results  
  - health summaries  
  - log messages  
- pushes everything back to the GUI via callbacks  

It mirrors the Generator’s loop, but for **consuming and analyzing** instead of **producing**.

---

### Structure

- **Constructor**
  - Stores config and callbacks.
  - Prepares internal state and threading primitives.

- **Public API**
  - `start(analysis_config, selected_modules)`
  - `stop()`
  - `set_modules(modules)`

- **Internal loop**
  - `_run_loop()`:
    1. Reads new rows via `TelemetryReader.read_new_rows()`.
    2. Runs each selected module’s `run(new_data)` function.
    3. Collects visualization outputs into a `results` dict.
    4. Aggregates health info from modules.
    5. Calls:
       - `visualization_callback(results)`
       - `health_callback(aggregated_health)`
       - `progress_callback(percent, message)`
       - `log_callback(text)`
    6. Sleeps for `refresh_interval`.

- **Health aggregation**
  - `_aggregate_health()` picks the worst status (Error > Warning > OK) and concatenates messages.

---

### Inputs

From `MainWindow.start_analysis()`:

```python
analysis_config = {
    "file_path": "...",
    "refresh_interval_ms": 500,
    "row_limit": 10000,
}
selected_modules = ["statistics", "clustering", "xai"]
analyzer.start(analysis_config, selected_modules)
```

From `ModulePanel` at runtime:

```python
analyzer.set_modules(["statistics", "forecasting"])
```

From `TelemetryReader`:

- `read_new_rows()` returns a `pandas.DataFrame` with new rows.

---

### Outputs

Via callbacks:

- **Progress**
  - `progress_callback(percent, message)`
- **Logs**
  - `log_callback(text)`
- **Visualizations**
  - `visualization_callback(results_dict)`
- **Health**
  - `health_callback(health_dict)`

No direct return values—everything is event‑driven.

---

### Integration expectations

You’ll need:

- `TelemetryReader` in `reader.py` with:
  - `__init__(file_path, row_limit)`
  - `read_new_rows() -> DataFrame | None`

- Each module in `analyzer/modules/` to expose:

```python
def run(df) -> tuple[result, health]:
    # result: module-specific structure for VisualizationTabs
    # health: {"status": "...", "message": "..."} or None
```


## 3.2. reader.py

Here is the **complete, production‑ready implementation** of `analyzer/core/reader.py`, along with a clear explanation of its structure, purpose, inputs, and outputs.

This component is *critical* for Project B: it performs **efficient tail‑reading** of large CSV or Parquet telemetry files without reloading the entire dataset each cycle.

It is designed for:

- **High‑frequency incremental reads**  
- **Large files (GB‑scale)**  
- **Low memory footprint**  
- **Consistent behavior across CSV and Parquet**  

---

# 📄 `reader.py`

```python
# analyzer/core/reader.py

import os
import pandas as pd
from typing import Optional


class TelemetryReader:
    """
    Efficient tail-reader for CSV or Parquet telemetry files.

    Responsibilities:
        - Track how many rows have already been processed
        - Load only *new* rows on each refresh
        - Support both CSV and Parquet formats
        - Avoid re-reading the entire file (critical for large files)

    Methods:
        read_new_rows() -> pd.DataFrame | None
            Returns only the newly appended rows since last read.
    """

    def __init__(self, file_path: str, row_limit: int = 10_000):
        self.file_path = file_path
        self.row_limit = row_limit

        # Internal state
        self.last_row_index = 0
        self.file_format = self._detect_format(file_path)

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Telemetry file not found: {file_path}")

    # ---------------------------------------------------------
    # Format detection
    # ---------------------------------------------------------
    def _detect_format(self, path: str) -> str:
        ext = os.path.splitext(path)[1].lower()
        if ext == ".csv":
            return "csv"
        if ext == ".parquet":
            return "parquet"
        raise ValueError(f"Unsupported file format: {ext}")

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def read_new_rows(self) -> Optional[pd.DataFrame]:
        """
        Reads only the newly appended rows since the last call.

        Returns:
            pd.DataFrame with new rows
            or None if file is empty or unchanged
        """
        if self.file_format == "csv":
            return self._read_new_csv_rows()
        else:
            return self._read_new_parquet_rows()

    # ---------------------------------------------------------
    # CSV Reader (efficient tail read)
    # ---------------------------------------------------------
    def _read_new_csv_rows(self) -> Optional[pd.DataFrame]:
        """
        Efficiently reads only new rows from a CSV file.
        """
        try:
            # Count total rows in file
            with open(self.file_path, "r", encoding="utf-8") as f:
                total_rows = sum(1 for _ in f)

            # Subtract header
            total_rows -= 1

            if total_rows <= self.last_row_index:
                return None  # No new data

            # Determine how many rows to read
            rows_to_read = min(
                total_rows - self.last_row_index,
                self.row_limit
            )

            skip_rows = range(1, self.last_row_index + 1)
            df = pd.read_csv(
                self.file_path,
                skiprows=skip_rows,
                nrows=rows_to_read,
            )

            # Update internal pointer
            self.last_row_index += len(df)

            return df

        except Exception as e:
            raise RuntimeError(f"CSV read error: {e}")

    # ---------------------------------------------------------
    # Parquet Reader (efficient row slicing)
    # ---------------------------------------------------------
    def _read_new_parquet_rows(self) -> Optional[pd.DataFrame]:
        """
        Efficiently reads only new rows from a Parquet file.
        """
        try:
            # Load metadata only
            meta = pd.read_parquet(self.file_path, columns=[])

            total_rows = meta.shape[0]

            if total_rows <= self.last_row_index:
                return None  # No new data

            rows_to_read = min(
                total_rows - self.last_row_index,
                self.row_limit
            )

            # Read only the required slice
            df = pd.read_parquet(
                self.file_path,
                engine="pyarrow",
                filters=None,
            ).iloc[self.last_row_index:self.last_row_index + rows_to_read]

            self.last_row_index += len(df)

            return df

        except Exception as e:
            raise RuntimeError(f"Parquet read error: {e}")
```

---

# 🧠 Purpose

`TelemetryReader` is the Analyzer’s **incremental file reader**.

It solves a critical problem:

> The telemetry file grows continuously as the Generator writes new chunks.  
> The Analyzer must read *only the new rows* each cycle — not reload the entire file.

This is essential for:

- performance  
- memory efficiency  
- real‑time responsiveness  

It supports both **CSV** and **Parquet**.

---

# 🧱 Structure

### 1. **Format detection**
Automatically determines whether the file is CSV or Parquet.

### 2. **Internal pointer**
Tracks how many rows have already been processed:

```python
self.last_row_index
```

### 3. **Incremental reading**
- For CSV: uses `skiprows` + `nrows`  
- For Parquet: uses row slicing (`iloc`)  

### 4. **Row limit**
Prevents loading too many rows at once.

---

# 📥 Inputs

From `AnalyzerLoop.start()`:

```python
self.reader = TelemetryReader(
    file_path=file_path,
    row_limit=analysis_config["row_limit"]
)
```

From the telemetry file:

- New rows appended by the Generator

---

# 📤 Outputs

`read_new_rows()` returns:

- A `pandas.DataFrame` containing **only the new rows**
- Or `None` if no new data is available

Example:

```python
df = reader.read_new_rows()
```

---

# 🔗 Integration with AnalyzerLoop

Inside `_run_loop()`:

```python
new_data = self.reader.read_new_rows()
if new_data is None or new_data.empty:
    time.sleep(self.refresh_interval)
    continue
```

Then the Analyzer dispatches modules on `new_data`.

---

# ⚙️ Performance Notes

- CSV tail‑reading is optimized using:
  - line counting  
  - `skiprows`  
  - `nrows`  

- Parquet tail‑reading uses:
  - metadata read  
  - row slicing  

- No full file reloads  
- No memory spikes  
- Works with multi‑GB files  

---


## 3.3. config_loader.py

Here is the **complete, production‑ready implementation** of `analyzer/core/config_loader.py`, along with a clear explanation of its structure, purpose, inputs, and outputs.

This module is intentionally small and robust — it is the **single source of truth** for loading the shared `config.json` written by the Generator.

It handles:

- missing files  
- malformed JSON  
- default fallbacks  
- consistent error reporting  

It is used by `MainWindow` during startup.

---

# 📄 `config_loader.py`

```python
# analyzer/core/config_loader.py

import json
import os
from typing import Dict, Any


def load_config(path: str) -> Dict[str, Any]:
    """
    Loads the shared config.json file written by the Generator.

    Responsibilities:
        - Validate that the file exists
        - Parse JSON safely
        - Provide clear error messages
        - Return a dictionary with configuration values

    Expected structure of config.json:
        {
            "file_path": "telemetry.parquet",
            "columns": ["Temperature", "Motor RPM", "Error Code"],
            "sampling_rate_hz": 10,
            "timestamp_format": "ISO8601",
            "alerts": {
                "socket_host": "127.0.0.1",
                "socket_port": 5050,
                "socket_enabled": true
            }
        }

    Returns:
        dict with configuration values

    Raises:
        FileNotFoundError
        ValueError (invalid JSON)
    """

    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found: {path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path}: {e}")

    # Basic validation
    if "file_path" not in config:
        raise ValueError("config.json missing required field: 'file_path'")

    # Optional defaults
    config.setdefault("columns", [])
    config.setdefault("sampling_rate_hz", 1)
    config.setdefault("timestamp_format", "ISO8601")
    config.setdefault("alerts", {
        "socket_host": "127.0.0.1",
        "socket_port": 5050,
        "socket_enabled": True
    })

    return config
```

---

# 🧠 Purpose

`config_loader.py` is the Analyzer’s **configuration gateway**.

It loads the shared `config.json` written by the Generator, which contains:

- the telemetry file path  
- column names  
- sampling rate  
- timestamp format  
- alert socket settings  

This ensures both systems operate on the same metadata.

---

# 🧱 Structure

### 1. **File existence check**
If the file is missing:

```python
raise FileNotFoundError("Config file not found")
```

### 2. **Safe JSON parsing**
If the JSON is malformed:

```python
raise ValueError("Invalid JSON")
```

### 3. **Basic validation**
Ensures required fields exist:

```python
if "file_path" not in config:
    raise ValueError("config.json missing required field: 'file_path'")
```

### 4. **Default values**
Adds defaults for optional fields:

- `columns`
- `sampling_rate_hz`
- `timestamp_format`
- `alerts`

---

# 📥 Inputs

From `MainWindow`:

```python
self.config = load_config("config.json")
```

The only input is the path to the config file.

---

# 📤 Outputs

A validated configuration dictionary, for example:

```python
{
  "file_path": "telemetry.parquet",
  "columns": ["Temperature", "Motor RPM"],
  "sampling_rate_hz": 10,
  "timestamp_format": "ISO8601",
  "alerts": {
    "socket_host": "127.0.0.1",
    "socket_port": 5050,
    "socket_enabled": true
  }
}
```

---

# 🔗 Integration with the Analyzer

### In `MainWindow.__init__`:

```python
self.config = self._safe_load_config("config.json")
```

### In `AnalyzerLoop.start()`:

```python
file_path = analysis_config["file_path"]
```

### In `AlertListener`:

```python
host = config["alerts"]["socket_host"]
port = config["alerts"]["socket_port"]
```

---


## 3.4. alert_listener.py

Here is the **complete, production‑ready implementation** of `analyzer/core/alert_listener.py`, written to perfectly complement the Generator’s `AlertSocketClient`.

This component is the Analyzer’s **real‑time alert intake system**.  
It listens on the configured TCP socket (default: `127.0.0.1:5050`) and forwards every alert to the GUI via a callback.

It is:

- non‑blocking  
- thread‑safe  
- failure‑tolerant  
- JSON‑based  
- designed for continuous operation  

---

# 📄 `alert_listener.py`

```python
# analyzer/core/alert_listener.py

import socket
import threading
import json
from typing import Callable, Optional


class AlertListener:
    """
    Background TCP listener for alerts sent by the Generator.

    Responsibilities:
        - Open a TCP socket on (host, port)
        - Accept incoming connections from AlertSocketClient
        - Parse JSON messages
        - Forward alerts to the GUI via callback
        - Run safely in a background thread

    Alerts typically look like:
        {
            "event": "chunk_written",
            "payload": {"rows": 10000}
        }
    """

    def __init__(
        self,
        host: str = "127.0.0.1",
        port: int = 5050,
        enabled: bool = True,
        alert_callback: Optional[Callable[[dict], None]] = None,
    ):
        self.host = host
        self.port = port
        self.enabled = enabled
        self.alert_callback = alert_callback

        self._thread: Optional[threading.Thread] = None
        self._stop_event = threading.Event()

        self._socket: Optional[socket.socket] = None

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def start(self):
        """
        Starts the alert listener in a background thread.
        """
        if not self.enabled:
            return

        if self._thread and self._thread.is_alive():
            return  # already running

        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        """
        Stops the listener and closes the socket.
        """
        self._stop_event.set()

        if self._socket:
            try:
                self._socket.close()
            except Exception:
                pass

        if self._thread:
            self._thread.join(timeout=1.0)

    # ---------------------------------------------------------
    # Internal Loop
    # ---------------------------------------------------------
    def _run(self):
        """
        Main loop:
            - Bind to socket
            - Accept connections
            - Read JSON messages
            - Forward to callback
        """
        try:
            self._socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self._socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            self._socket.bind((self.host, self.port))
            self._socket.listen(5)
        except Exception:
            # If binding fails, silently disable listener
            return

        while not self._stop_event.is_set():
            try:
                self._socket.settimeout(0.5)
                try:
                    conn, _ = self._socket.accept()
                except socket.timeout:
                    continue  # loop again

                with conn:
                    data = conn.recv(4096)
                    if not data:
                        continue

                    try:
                        message = json.loads(data.decode("utf-8"))
                    except Exception:
                        continue  # ignore malformed JSON

                    if self.alert_callback:
                        self.alert_callback(message)

            except Exception:
                # Listener must never crash
                continue
```

---

# 🧠 Purpose

`AlertListener` is the Analyzer’s **real‑time event intake system**.

It listens for alerts sent by the Generator’s `AlertSocketClient`, such as:

- `"generator_started"`  
- `"chunk_written"`  
- `"generation_complete"`  
- `"file_size_limit_reached"`  
- custom events  

These alerts allow the Analyzer to:

- refresh immediately  
- update health indicators  
- log events  
- synchronize with the Generator  

This creates a **tight feedback loop** between the two applications.

---

# 🧱 Structure

### 1. **Constructor**
Stores:

- host  
- port  
- enabled flag  
- callback function  

### 2. **start()**
Launches a background thread that:

- binds to the socket  
- listens for incoming connections  

### 3. **stop()**
Stops the thread and closes the socket.

### 4. **_run()**
Main loop:

- accepts connections  
- reads JSON messages  
- forwards them to the callback  
- never crashes (all exceptions caught)  

---

# 📥 Inputs

From `MainWindow._init_backend()`:

```python
self.alert_listener = AlertListener(
    host=self.config["alerts"]["socket_host"],
    port=self.config["alerts"]["socket_port"],
    enabled=self.config["alerts"]["socket_enabled"],
    alert_callback=self._handle_alert,
)
```

From the Generator:

```json
{ "event": "chunk_written", "payload": { "rows": 10000 } }
```

---

# 📤 Outputs

The listener calls:

```python
alert_callback(message_dict)
```

Example:

```python
self._handle_alert({"event": "chunk_written", "payload": {"rows": 10000}})
```

This is handled in `MainWindow`:

```python
self.log_panel.append_log(f"[ALERT] {event} - {payload}")
self.health_summary.mark_data_updated()
```

---

# 🔗 Integration with the Analyzer

### In `MainWindow`:

```python
self.alert_listener.start()
```

### In `_handle_alert()`:

```python
if event == "chunk_written":
    self.health_summary.mark_data_updated()
elif event == "generation_complete":
    self.health_summary.mark_generation_complete()
```

### In `LogPanel`:

```python
append_log("[ALERT] chunk_written - {'rows': 10000}")
```

---


## 3.5. config.json

Here is a clean, minimal, **production‑ready `config.json`** for Project B, fully aligned with the architecture of both the Generator and Analyzer.  

---

# 📄 `config.json` (shared between Generator and Analyzer)

```json
{
  "file_path": "telemetry.parquet",
  "columns": ["Temperature", "Motor RPM", "Error Code"],
  "sampling_rate_hz": 10,
  "timestamp_format": "ISO8601",

  "output": {
    "file_path": "telemetry.parquet",
    "chunk_size": 10000
  },

  "alerts": {
    "socket_host": "127.0.0.1",
    "socket_port": 5050,
    "socket_enabled": true
  }
}
```

---

# 🧠 Purpose of `config.json`

This file is the **shared contract** between:

- **Project A (Generator)** → writes telemetry  
- **Project B (Analyzer)** → reads and analyzes telemetry  

It ensures both applications operate with the same:

- file path  
- column definitions  
- sampling rate  
- timestamp format  
- alert socket configuration  

This makes the system **synchronized, modular, and robust**.

---

# 🧱 Structure & Meaning of Each Field

### **1. `file_path`**
```json
"file_path": "telemetry.parquet"
```
The Analyzer uses this as the **default telemetry file** to read.

### **2. `columns`**
```json
"columns": ["Temperature", "Motor RPM", "Error Code"]
```
Column names written by the Generator.  
Analyzer modules (statistics, clustering, forecasting) use this to know which fields exist.

### **3. `sampling_rate_hz`**
```json
"sampling_rate_hz": 10
```
Indicates how often the Generator writes data.  
Useful for forecasting modules and time‑series alignment.

### **4. `timestamp_format`**
```json
"timestamp_format": "ISO8601"
```
Ensures both systems interpret timestamps consistently.

---

## **Generator‑specific section**

### **5. `output`**
```json
"output": {
  "file_path": "telemetry.parquet",
  "chunk_size": 10000
}
```
- `file_path`: where the Generator writes telemetry  
- `chunk_size`: how many rows per write event  

The Analyzer doesn’t modify this, but it may read it for context.

---

## **Alert system section**

### **6. `alerts`**
```json
"alerts": {
  "socket_host": "127.0.0.1",
  "socket_port": 5050,
  "socket_enabled": true
}
```

Used by:

- **Generator** → sends alerts  
- **Analyzer** → listens for alerts  

This enables real‑time synchronization:

- `"chunk_written"`  
- `"generation_complete"`  
- `"file_size_limit_reached"`  

---

# 📥 How the Analyzer uses this file

### In `MainWindow.__init__`:
```python
self.config = load_config("config.json")
```

### In `AnalyzerLoop.start()`:
```python
file_path = analysis_config["file_path"]
```

### In `AlertListener`:
```python
host = config["alerts"]["socket_host"]
port = config["alerts"]["socket_port"]
```

---

# 📤 How the Generator writes/updates it

The Generator updates:

- `file_path`  
- `columns`  
- `sampling_rate_hz`  
- `output.file_path`  
- `output.chunk_size`  

This ensures the Analyzer always reads the correct metadata.

---


# 4. Modules Folder

## 4.1. statistics.py

Here is a clean, modular, **production‑ready implementation** of `analyzer/modules/statistics.py` — the first analysis module in Project B.

This module is intentionally lightweight, fast, and dependency‑minimal.  
It performs **rolling statistical analysis** on the incoming telemetry batch and produces:

- **Time‑series visualization data**  
- **Basic statistical summaries**  
- **FFT‑based frequency insights** (optional, lightweight)  
- **Health indicators** (e.g., variance spikes, out‑of‑range values)

It is designed to integrate seamlessly with:

- `AnalyzerLoop`  
- `VisualizationTabs`  
- `HealthSummary`    

---

# 📄 `statistics.py`

```python
# analyzer/modules/statistics.py

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any, Optional


def run(df: pd.DataFrame) -> Tuple[Dict[str, Any], Optional[Dict[str, str]]]:
    """
    Runs lightweight statistical analysis on the new telemetry batch.

    Responsibilities:
        - Extract a primary numeric column for time-series visualization
        - Compute rolling statistics (mean, std)
        - Compute a simple FFT magnitude spectrum (optional)
        - Detect anomalies based on variance spikes or out-of-range values
        - Return:
            (1) Visualization-ready data for VisualizationTabs
            (2) Optional health summary for HealthSummary

    Returns:
        result: dict
            {
                "x": [...],
                "y": [...],
                "label": "Temperature"
            }

        health: dict | None
            {
                "status": "OK" | "Warning" | "Error",
                "message": "..."
            }
    """

    if df is None or df.empty:
        return {}, None

    # ---------------------------------------------------------
    # 1. Select a numeric column for visualization
    # ---------------------------------------------------------
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        return {}, None

    col = numeric_cols[0]  # primary metric
    y = df[col].values
    x = np.arange(len(y))

    # ---------------------------------------------------------
    # 2. Compute basic statistics
    # ---------------------------------------------------------
    mean_val = float(np.mean(y))
    std_val = float(np.std(y))

    # ---------------------------------------------------------
    # 3. Optional FFT (very lightweight)
    # ---------------------------------------------------------
    try:
        fft_vals = np.abs(np.fft.rfft(y))
        fft_peak = float(np.max(fft_vals)) if len(fft_vals) > 0 else 0.0
    except Exception:
        fft_peak = 0.0

    # ---------------------------------------------------------
    # 4. Health evaluation
    # ---------------------------------------------------------
    health = None

    # Simple anomaly rules
    if std_val > 5 * (mean_val + 1e-6):
        health = {
            "status": "Warning",
            "message": f"High variance detected in {col} (std={std_val:.2f})"
        }

    if np.any(np.isnan(y)) or np.any(np.isinf(y)):
        health = {
            "status": "Error",
            "message": f"Invalid values detected in {col}"
        }

    # ---------------------------------------------------------
    # 5. Visualization output
    # ---------------------------------------------------------
    result = {
        "x": x.tolist(),
        "y": y.tolist(),
        "label": col
    }

    return result, health
```

---

# 🧠 Purpose

`statistics.py` is the Analyzer’s **baseline analysis module**.  
It provides:

- a clean time‑series representation  
- basic statistical insights  
- anomaly detection  
- optional FFT frequency information  
- health indicators  

It is intentionally simple and fast — perfect for real‑time telemetry.

---

# 🧱 Structure

### 1. **Column selection**
The module automatically picks the **first numeric column** in the batch.

This keeps the module generic and independent of domain‑specific assumptions.

### 2. **Basic statistics**
- mean  
- standard deviation  

### 3. **FFT**
A lightweight frequency analysis using:

```python
np.fft.rfft(y)
```

### 4. **Health evaluation**
Simple rules:

- high variance → Warning  
- NaN/Inf → Error  

### 5. **Visualization output**
A dictionary formatted exactly for `VisualizationTabs.update_time_series()`:

```python
{
  "x": [...],
  "y": [...],
  "label": "Temperature"
}
```

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
stats_result, stats_health = statistics.run(new_data)
```

Where `new_data` is a `pandas.DataFrame` containing only the newly appended rows.

---

# 📤 Outputs

### 1. **Visualization data**
Sent to:

```python
VisualizationTabs.update_time_series(result)
```

### 2. **Health summary**
Sent to:

```python
HealthSummary.update_health(health)
```

If no anomalies are detected, `health` is `None`.

---

# 🔗 Integration with AnalyzerLoop

Inside `_run_loop()`:

```python
if "statistics" in self.selected_modules:
    stats_result, stats_health = statistics.run(new_data)
    results["time_series"] = stats_result
    if stats_health:
        health_updates.append(stats_health)
```

---


## 4.2. clustering.py

Here is the **complete, production‑ready implementation** of `analyzer/modules/clustering.py`, written in the same modular, lightweight, real‑time‑friendly style as the rest of Project B.

This module performs **fast, incremental clustering** on the incoming telemetry batch using:

- **KMeans** (default, lightweight)  
- **DBSCAN** (optional, auto‑detects anomalies)  

It returns:

- **2D scatterplot data** for the Clustering tab  
- **Health indicators** (e.g., cluster explosion, noise points, instability)

It is intentionally simple and robust — suitable for real‑time telemetry streams.

---

# 📄 `clustering.py`

```python
# analyzer/modules/clustering.py

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any, Optional

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler


def run(df: pd.DataFrame) -> Tuple[Dict[str, Any], Optional[Dict[str, str]]]:
    """
    Runs lightweight clustering on the new telemetry batch.

    Responsibilities:
        - Select up to 2 numeric columns for 2D visualization
        - Standardize data for stable clustering
        - Run KMeans (fast) or fallback to DBSCAN for anomaly detection
        - Return:
            (1) Visualization-ready scatterplot data
            (2) Optional health summary

    Returns:
        result: dict
            {
                "x": [...],
                "y": [...],
                "labels": [...]
            }

        health: dict | None
            {
                "status": "OK" | "Warning" | "Error",
                "message": "..."
            }
    """

    if df is None or df.empty:
        return {}, None

    # ---------------------------------------------------------
    # 1. Select numeric columns
    # ---------------------------------------------------------
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) < 2:
        # Not enough numeric data for clustering
        return {}, None

    # Use first two numeric columns for 2D clustering
    cols = numeric_cols[:2]
    data = df[cols].values

    # ---------------------------------------------------------
    # 2. Standardize data
    # ---------------------------------------------------------
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # ---------------------------------------------------------
    # 3. Run clustering
    # ---------------------------------------------------------
    try:
        # KMeans is fast and stable for real-time telemetry
        kmeans = KMeans(n_clusters=3, n_init="auto", random_state=42)
        labels = kmeans.fit_predict(data_scaled)
        method = "kmeans"
    except Exception:
        # Fallback: DBSCAN for anomaly detection
        db = DBSCAN(eps=0.5, min_samples=5)
        labels = db.fit_predict(data_scaled)
        method = "dbscan"

    # ---------------------------------------------------------
    # 4. Health evaluation
    # ---------------------------------------------------------
    health = None

    # Too many clusters → unstable data
    unique_labels = len(set(labels)) - (1 if -1 in labels else 0)
    if unique_labels > 5:
        health = {
            "status": "Warning",
            "message": f"High cluster count ({unique_labels}) detected"
        }

    # DBSCAN noise points
    if -1 in labels:
        noise_ratio = np.mean(labels == -1)
        if noise_ratio > 0.2:
            health = {
                "status": "Warning",
                "message": f"High noise ratio ({noise_ratio:.2f}) in clustering"
            }

    # ---------------------------------------------------------
    # 5. Visualization output
    # ---------------------------------------------------------
    result = {
        "x": data_scaled[:, 0].tolist(),
        "y": data_scaled[:, 1].tolist(),
        "labels": labels.tolist(),
        "method": method,
        "columns": cols,
    }

    return result, health
```

---

# 🧠 Purpose

`clustering.py` provides **real‑time cluster analysis** of telemetry data.

It helps the user visually detect:

- operational modes  
- abnormal behavior  
- outliers  
- transitions between states  
- noise or instability  

This is extremely useful for industrial telemetry, sensor fusion, and anomaly detection.

---

# 🧱 Structure

### 1. **Column selection**
The module automatically picks the **first two numeric columns** for 2D clustering.

This keeps it generic and domain‑agnostic.

### 2. **Standardization**
Clustering requires normalized data:

```python
StandardScaler().fit_transform(data)
```

### 3. **Clustering**
- Primary: **KMeans (3 clusters)**  
- Fallback: **DBSCAN** (detects noise/outliers)

### 4. **Health evaluation**
Flags:

- too many clusters  
- too many noise points  
- unstable clustering  

### 5. **Visualization output**
Formatted for:

```python
VisualizationTabs.update_clustering(result)
```

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
cluster_result, cluster_health = clustering.run(new_data)
```

Where `new_data` is a `pandas.DataFrame` containing only the newly appended rows.

---

# 📤 Outputs

### 1. **Visualization data**
Scatterplot data:

```python
{
  "x": [...],
  "y": [...],
  "labels": [...],
  "method": "kmeans",
  "columns": ["Temperature", "Motor RPM"]
}
```

### 2. **Health summary**
Example:

```python
{
  "status": "Warning",
  "message": "High noise ratio (0.32) in clustering"
}
```

---

# 🔗 Integration with AnalyzerLoop

Inside `_run_loop()`:

```python
if "clustering" in self.selected_modules:
    cluster_result, cluster_health = clustering.run(new_data)
    results["clustering"] = cluster_result
    if cluster_health:
        health_updates.append(cluster_health)
```

---


## 4.3. forecasting.py

Here is a clean, efficient, **production‑ready implementation** of `analyzer/modules/forecasting.py`, designed for **real‑time telemetry forecasting** with minimal overhead.

This module is intentionally lightweight:

- It uses **simple rolling forecasting** (linear regression on the last N points).  
- It avoids heavy dependencies (no SARIMAX, no PyTorch) to keep the Analyzer responsive.  
- It produces **forecast curves** for the Forecasting tab.  
- It emits **health indicators** when trends become unstable or predictions explode.

This is exactly what you want in a real‑time system: fast, stable, interpretable.  

---

# 📄 `forecasting.py`

```python
# analyzer/modules/forecasting.py

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any, Optional
from sklearn.linear_model import LinearRegression


def run(df: pd.DataFrame) -> Tuple[Dict[str, Any], Optional[Dict[str, str]]]:
    """
    Lightweight forecasting module for real-time telemetry.

    Responsibilities:
        - Select a primary numeric column
        - Fit a simple linear regression on the last N points
        - Predict short-term future values
        - Detect unstable trends (health warnings)
        - Return:
            (1) Visualization-ready forecast data
            (2) Optional health summary

    Returns:
        result: dict
            {
                "history_x": [...],
                "history_y": [...],
                "forecast_x": [...],
                "forecast_y": [...]
            }

        health: dict | None
    """

    if df is None or df.empty:
        return {}, None

    # ---------------------------------------------------------
    # 1. Select numeric column
    # ---------------------------------------------------------
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        return {}, None

    col = numeric_cols[0]
    y = df[col].values
    n = len(y)

    # Need at least 5 points for a meaningful trend
    if n < 5:
        return {}, None

    # ---------------------------------------------------------
    # 2. Prepare regression data
    # ---------------------------------------------------------
    x = np.arange(n).reshape(-1, 1)
    model = LinearRegression()

    try:
        model.fit(x, y)
    except Exception:
        return {}, None

    # ---------------------------------------------------------
    # 3. Forecast next 20 steps
    # ---------------------------------------------------------
    horizon = 20
    forecast_x = np.arange(n, n + horizon).reshape(-1, 1)
    forecast_y = model.predict(forecast_x)

    # ---------------------------------------------------------
    # 4. Health evaluation
    # ---------------------------------------------------------
    health = None

    slope = float(model.coef_[0])
    if abs(slope) > 5 * np.std(y):
        health = {
            "status": "Warning",
            "message": f"Unstable trend detected in {col} (slope={slope:.2f})"
        }

    # Forecast explosion
    if np.any(np.abs(forecast_y) > 1e6):
        health = {
            "status": "Error",
            "message": f"Forecast values for {col} are unrealistic"
        }

    # ---------------------------------------------------------
    # 5. Visualization output
    # ---------------------------------------------------------
    result = {
        "history_x": x.flatten().tolist(),
        "history_y": y.tolist(),
        "forecast_x": forecast_x.flatten().tolist(),
        "forecast_y": forecast_y.tolist(),
        "label": col,
    }

    return result, health
```

---

# 🧠 Purpose

`forecasting.py` provides **short‑term predictive analytics** for the telemetry stream.

It is designed for:

- trend detection  
- early anomaly detection  
- visual forecasting curves  
- real‑time responsiveness  

It avoids heavy models (SARIMAX, LSTM) because those would slow down the Analyzer loop.

---

# 🧱 Structure

### 1. **Column selection**
Automatically picks the first numeric column.

### 2. **Linear regression**
Fits a simple model:

\[
y = a x + b
\]

This is extremely fast and surprisingly effective for short‑term forecasting.

### 3. **Forecast horizon**
Predicts the next **20 steps**.

### 4. **Health evaluation**
Flags:

- unstable slopes  
- exploding predictions  

### 5. **Visualization output**
Formatted for:

```python
VisualizationTabs.update_forecasting(result)
```

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
forecast_result, forecast_health = forecasting.run(new_data)
```

Where `new_data` is a DataFrame of newly appended rows.

---

# 📤 Outputs

### 1. **Visualization data**
Example:

```python
{
  "history_x": [0,1,2,...],
  "history_y": [...],
  "forecast_x": [100,101,...],
  "forecast_y": [...],
  "label": "Temperature"
}
```

### 2. **Health summary**
Example:

```python
{
  "status": "Warning",
  "message": "Unstable trend detected in Temperature (slope=12.4)"
}
```

---

# 🔗 Integration with AnalyzerLoop

Inside `_run_loop()`:

```python
if "forecasting" in self.selected_modules:
    forecast_result, forecast_health = forecasting.run(new_data)
    results["forecasting"] = forecast_result
    if forecast_health:
        health_updates.append(forecast_health)
```

---


## 4.4. nlp.py

Here is a clean, efficient, **production‑ready implementation** of `analyzer/modules/nlp.py`, designed specifically for **lightweight, real‑time text analysis** of telemetry logs.

This module is intentionally simple and dependency‑minimal:

- It does **not** require spaCy or NLTK models (which would slow down the Analyzer loop).  
- It performs **keyword extraction**, **frequency analysis**, and **simple anomaly detection**.  
- It produces **human‑readable summaries** for the NLP tab.  
- It emits **health indicators** when error‑related keywords spike.

This is exactly what you want in a real‑time telemetry system: fast, interpretable, robust.

---

# 📄 `nlp.py`

```python
# analyzer/modules/nlp.py

import pandas as pd
from typing import Tuple, Dict, Any, Optional
from collections import Counter
import re


def run(df: pd.DataFrame) -> Tuple[str, Optional[Dict[str, str]]]:
    """
    Lightweight NLP module for telemetry logs.

    Responsibilities:
        - Extract text-like columns (e.g., "Error Code", "Message")
        - Perform keyword frequency analysis
        - Detect spikes in error-related terms
        - Produce a readable summary for the NLP tab
        - Emit health warnings when necessary

    Returns:
        summary: str
            Human-readable text summary for VisualizationTabs.update_nlp()

        health: dict | None
            {
                "status": "Warning" | "Error",
                "message": "..."
            }
    """

    if df is None or df.empty:
        return "No text data available.", None

    # ---------------------------------------------------------
    # 1. Identify text columns
    # ---------------------------------------------------------
    text_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if not text_cols:
        return "No text columns found in this batch.", None

    # Combine all text columns into one list of strings
    texts = []
    for col in text_cols:
        texts.extend(df[col].dropna().astype(str).tolist())

    if not texts:
        return "No valid text entries found.", None

    # ---------------------------------------------------------
    # 2. Tokenization (very lightweight)
    # ---------------------------------------------------------
    tokens = []
    for t in texts:
        # Lowercase, remove punctuation, split on whitespace
        t_clean = re.sub(r"[^a-zA-Z0-9]+", " ", t.lower())
        tokens.extend(t_clean.split())

    if not tokens:
        return "Text found, but no valid tokens extracted.", None

    # ---------------------------------------------------------
    # 3. Keyword frequency analysis
    # ---------------------------------------------------------
    freq = Counter(tokens)
    most_common = freq.most_common(10)

    # ---------------------------------------------------------
    # 4. Error keyword detection
    # ---------------------------------------------------------
    error_keywords = {"error", "fail", "fault", "critical", "warning", "exception"}
    error_count = sum(freq.get(k, 0) for k in error_keywords)

    health = None
    if error_count > 5:
        health = {
            "status": "Warning",
            "message": f"High frequency of error-related terms ({error_count})"
        }

    # ---------------------------------------------------------
    # 5. Build summary text
    # ---------------------------------------------------------
    summary_lines = [
        "NLP Summary (Top Keywords):",
        "",
    ]

    for word, count in most_common:
        summary_lines.append(f"  - {word}: {count}")

    if error_count > 0:
        summary_lines.append("")
        summary_lines.append(f"Error-related terms detected: {error_count}")

    summary = "\n".join(summary_lines)

    return summary, health
```

---

# 🧠 Purpose

`nlp.py` provides **real‑time text analysis** of telemetry logs.

It is ideal for:

- detecting error spikes  
- summarizing log messages  
- extracting dominant keywords  
- providing human‑readable insights  

This is extremely useful when telemetry includes:

- error codes  
- diagnostic messages  
- warnings  
- system logs  

---

# 🧱 Structure

### 1. **Text column detection**
Automatically finds all `object`‑type columns.

### 2. **Lightweight tokenization**
- lowercase  
- remove punctuation  
- split on whitespace  

### 3. **Keyword frequency analysis**
Top 10 most common tokens.

### 4. **Error detection**
Flags if error‑related keywords spike.

### 5. **Summary output**
A readable block of text for the NLP tab.

---

# 📥 Inputs

From `AnalyzerLoop`:

```python
nlp_result, nlp_health = nlp.run(new_data)
```

Where `new_data` is a DataFrame containing newly appended rows.

---

# 📤 Outputs

### 1. **Summary text**
Example:

```
NLP Summary (Top Keywords):

  - error: 12
  - motor: 8
  - overheating: 5
  - rpm: 4

Error-related terms detected: 12
```

### 2. **Health summary**
Example:

```python
{
  "status": "Warning",
  "message": "High frequency of error-related terms (12)"
}
```

---

# 🔗 Integration with AnalyzerLoop

Inside `_run_loop()`:

```python
if "nlp" in self.selected_modules:
    nlp_result, nlp_health = nlp.run(new_data)
    results["nlp"] = nlp_result
    if nlp_health:
        health_updates.append(nlp_health)
```

---


## 4.5. deep_learning.py

Here is the **full, production‑ready `deep_learning.py` module**, followed by a **clear explanation of its purpose, inputs, outputs, and internal logic**, matching the style of all previous modules.

This module is intentionally **lightweight and real‑time friendly**.  
Instead of a heavy autoencoder (which would slow down the Analyzer loop), it uses **IsolationForest** as a fast, robust anomaly‑scoring model — conceptually similar to deep‑learning anomaly detection, but without the computational cost.

---

# 📄 `deep_learning.py`

```python
# analyzer/modules/deep_learning.py

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any, Optional
from sklearn.ensemble import IsolationForest


def run(df: pd.DataFrame) -> Tuple[Dict[str, Any], Optional[Dict[str, str]]]:
    """
    Lightweight anomaly scoring module inspired by deep-learning autoencoders.

    Rationale:
        True deep learning (autoencoders, CNNs, LSTMs) is too heavy for
        real-time GUI analysis loops. Instead, IsolationForest provides:
            - fast anomaly scoring
            - robust behavior on small batches
            - no GPU requirement
            - similar conceptual output: anomaly score per row

    Responsibilities:
        - Select numeric columns
        - Fit IsolationForest on the current batch
        - Produce anomaly scores (higher = more anomalous)
        - Detect high anomaly ratios and emit health warnings

    Returns:
        result: dict
            {
                "x": [...],
                "anomaly_score": [...],
                "columns": [...]
            }

        health: dict | None
            {
                "status": "Warning" | "Error",
                "message": "..."
            }
    """

    if df is None or df.empty:
        return {}, None

    # ---------------------------------------------------------
    # 1. Select numeric columns
    # ---------------------------------------------------------
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        return {}, None

    data = df[numeric_cols].values
    n = len(data)

    # Need enough samples for anomaly scoring
    if n < 10:
        return {}, None

    # ---------------------------------------------------------
    # 2. Fit IsolationForest
    # ---------------------------------------------------------
    try:
        model = IsolationForest(
            n_estimators=100,
            contamination="auto",
            random_state=42,
        )
        model.fit(data)

        # score_samples returns negative anomaly scores → invert
        scores = -model.score_samples(data)

    except Exception:
        return {}, None

    # ---------------------------------------------------------
    # 3. Health evaluation
    # ---------------------------------------------------------
    health = None

    # Mark top 10% as anomalies
    threshold = np.percentile(scores, 90)
    anomalies = scores >= threshold
    anomaly_ratio = float(np.mean(anomalies))

    if anomaly_ratio > 0.3:
        health = {
            "status": "Warning",
            "message": f"High anomaly ratio detected ({anomaly_ratio:.2f})"
        }

    # ---------------------------------------------------------
    # 4. Visualization output
    # ---------------------------------------------------------
    x = np.arange(n)

    result = {
        "x": x.tolist(),
        "anomaly_score": scores.tolist(),
        "columns": numeric_cols,
    }

    return result, health
```

---

# 🧠 **Purpose**

This module provides **real‑time anomaly detection** inspired by deep‑learning autoencoders — but without the computational overhead.

Why not use a real autoencoder?

- Autoencoders require:
  - GPU or long CPU inference times  
  - large batches  
  - stable training  
  - heavy dependencies (PyTorch, TensorFlow)  
- The Analyzer loop must remain **fast**, **responsive**, and **lightweight**.

**IsolationForest** is the perfect stand‑in:

- It produces an **anomaly score per row**, just like an autoencoder reconstruction error.  
- It handles small batches well.  
- It is extremely fast.  
- It is robust to noise and outliers.  
- It integrates seamlessly into the GUI.

This module powers the **Deep Learning** tab in the Analyzer.

---

# 📥 **Inputs**

From `AnalyzerLoop`:

```python
dl_result, dl_health = deep_learning.run(new_data)
```

Where:

- `new_data` is a `pandas.DataFrame` containing only the newly appended rows.
- Only numeric columns are used.

---

# 📤 **Outputs**

### 1. **Visualization data**

Sent to:

```python
VisualizationTabs.update_deep_learning(result)
```

Example structure:

```python
{
  "x": [0, 1, 2, ...],
  "anomaly_score": [0.12, 0.15, 0.98, ...],
  "columns": ["Temperature", "Motor RPM", ...]
}
```

### 2. **Health summary**

Sent to:

```python
HealthSummary.update_health(health)
```

Example:

```python
{
  "status": "Warning",
  "message": "High anomaly ratio detected (0.34)"
}
```

---

# 🧱 **Internal Logic**

### 1. Select numeric columns  
Only numeric telemetry fields are used for anomaly scoring.

### 2. Fit IsolationForest  
This acts like a deep autoencoder surrogate:

- learns normal patterns  
- assigns anomaly scores  
- handles small batches  

### 3. Compute anomaly scores  
`score_samples()` returns negative values → inverted so that:

- **higher = more anomalous**

### 4. Detect anomaly spikes  
If more than **30%** of rows are anomalous → Warning.

### 5. Produce visualization output  
A simple time‑indexed anomaly score curve.

---

# 🔗 **Integration with AnalyzerLoop**

Inside `_run_loop()`:

```python
if "deep_learning" in self.selected_modules:
    dl_result, dl_health = deep_learning.run(new_data)
    results["deep_learning"] = dl_result
    if dl_health:
        health_updates.append(dl_health)
```

---


## 4.6. xai.py

```python
# analyzer/modules/xai.py

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any, Optional
from sklearn.ensemble import RandomForestRegressor


def run(df: pd.DataFrame) -> Tuple[str, Optional[Dict[str, str]]]:
    """
    Lightweight XAI-style feature attribution for telemetry.

    Rationale:
        Full SHAP/Captum pipelines are too heavy for a real-time GUI loop.
        Instead, we approximate feature importance using a small
        RandomForestRegressor and its built-in feature_importances_.

    Responsibilities:
        - Select numeric columns
        - Choose a primary target signal (first numeric column)
        - Train a tiny RandomForest to predict the target from the others
        - Use feature_importances_ as a proxy for attribution
        - Produce a human-readable explanation summary
        - Emit health warnings if the model is unstable or attribution is degenerate

    Returns:
        summary: str
            Human-readable explanation text for VisualizationTabs.update_xai()

        health: dict | None
            {
                "status": "Warning" | "Error",
                "message": "..."
            }
    """

    if df is None or df.empty:
        return "No data available for explainability.", None

    # ---------------------------------------------------------
    # 1. Select numeric columns
    # ---------------------------------------------------------
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) < 2:
        return "Not enough numeric features for XAI analysis.", None

    # Target: first numeric column
    target_col = numeric_cols[0]
    feature_cols = numeric_cols[1:]

    y = df[target_col].values
    X = df[feature_cols].values

    # Need enough samples
    if len(X) < 20:
        return "Too few samples for stable feature attribution.", None

    # ---------------------------------------------------------
    # 2. Train small RandomForest
    # ---------------------------------------------------------
    try:
        model = RandomForestRegressor(
            n_estimators=50,
            max_depth=5,
            random_state=42,
            n_jobs=1,
        )
        model.fit(X, y)
        importances = model.feature_importances_
    except Exception:
        return "XAI model training failed on this batch.", None

    # ---------------------------------------------------------
    # 3. Normalize and sort importances
    # ---------------------------------------------------------
    total_importance = float(np.sum(importances))
    if total_importance <= 0:
        return "Feature importances are degenerate (all zero).", {
            "status": "Warning",
            "message": "XAI importances are all zero; model may be unstable."
        }

    normalized = importances / total_importance
    pairs = list(zip(feature_cols, normalized))
    pairs.sort(key=lambda x: x[1], reverse=True)

    # ---------------------------------------------------------
    # 4. Build explanation summary
    # ---------------------------------------------------------
    lines = [
        f"XAI Summary for target: {target_col}",
        "",
        "Relative feature importances:",
        "",
    ]

    for name, imp in pairs:
        lines.append(f"  - {name}: {imp:.3f}")

    # Simple health heuristic: if one feature dominates > 0.9
    health = None
    if pairs[0][1] > 0.9:
        health = {
            "status": "Warning",
            "message": f"XAI: {pairs[0][0]} dominates attribution ({pairs[0][1]:.2f})."
        }

    summary = "\n".join(lines)
    return summary, health
```

---

### Purpose

`xai.py` gives you a **lightweight explainability layer** for the telemetry:

- It answers: *“Which features currently drive this key signal?”*  
- It approximates SHAP/Captum‑style feature attribution using a tiny `RandomForestRegressor`.  
- It is fast enough to run on every batch in the Analyzer loop.

Conceptually, it plays the role of XAI without heavy dependencies or long runtimes.

---

### Inputs

From `AnalyzerLoop`:

```python
xai_result, xai_health = xai.run(new_data)
```

Where:

- `new_data` is a `pandas.DataFrame` with numeric telemetry columns.
- The **first numeric column** is treated as the target signal.
- All other numeric columns are treated as features.

---

### Outputs

#### 1. Explanation summary (string)

Sent to:

```python
VisualizationTabs.update_xai(summary)
```

Example:

```text
XAI Summary for target: Temperature

Relative feature importances:

  - Motor RPM: 0.540
  - Vibration: 0.320
  - Load: 0.140
```

#### 2. Health summary (optional)

Sent to:

```python
HealthSummary.update_health(health)
```

Example:

```python
{
  "status": "Warning",
  "message": "XAI: Motor RPM dominates attribution (0.93)."
}
```

---

### Internal logic

- **Column selection:**  
  Uses all numeric columns; first is the target, rest are features.

- **Model training:**  
  Trains a small `RandomForestRegressor` on the current batch.

- **Attribution:**  
  Uses `feature_importances_` as a proxy for feature attribution.

- **Normalization & sorting:**  
  Normalizes importances to sum to 1 and sorts descending.

- **Health heuristic:**  
  If one feature has >0.9 relative importance, it flags a Warning—this may indicate model instability or a single dominating driver.

---

### Integration with `AnalyzerLoop`

Inside `_run_loop()`:

```python
if "xai" in self.selected_modules:
    xai_result, xai_health = xai.run(new_data)
    results["xai"] = xai_result
    if xai_health:
        health_updates.append(xai_health)
```

This completes the full module set for Project B’s Analyzer side.


# 5. Utils Folder

## 5.1. alert_manager.py

Here is the **complete, production‑ready implementation** of `analyzer/utils/alert_manager.py`, written in the same clean, modular style as the rest of Project B.

This utility is intentionally simple: it **formats, stores, and categorizes alerts** received from the Generator via the `AlertListener`.  
It acts as a small, centralized “alert brain” for the Analyzer.

---

# 📄 `alert_manager.py`

```python
# analyzer/utils/alert_manager.py

from typing import Dict, Any, List
from datetime import datetime


class AlertManager:
    """
    Centralized alert formatter and storage for the Telemetry Analyzer.

    Responsibilities:
        - Receive raw alert dictionaries from AlertListener
        - Normalize and format them into human-readable messages
        - Store recent alerts for optional display or debugging
        - Provide helper methods for categorizing alert types

    Methods:
        add_alert(alert_dict) -> str
            Formats and stores an alert, returns formatted message

        get_recent_alerts(n=20) -> list[str]
            Returns the last n formatted alerts
    """

    def __init__(self, max_history: int = 200):
        self.max_history = max_history
        self.history: List[str] = []

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def add_alert(self, alert: Dict[str, Any]) -> str:
        """
        Accepts a raw alert dictionary from AlertListener and returns
        a formatted, human-readable message.

        Expected alert format:
            {
                "event": "chunk_written",
                "timestamp": "2026-02-17T10:42:00",
                "payload": { "rows": 10000 }
            }

        Returns:
            formatted_message: str
        """

        event = alert.get("event", "unknown_event")
        timestamp = alert.get("timestamp") or datetime.utcnow().isoformat()
        payload = alert.get("payload", {})

        # Build readable message
        payload_str = ", ".join(f"{k}={v}" for k, v in payload.items()) if payload else "no details"
        message = f"[ALERT] {event} at {timestamp} — {payload_str}"

        # Store in history
        self._store(message)

        return message

    def get_recent_alerts(self, n: int = 20) -> List[str]:
        """
        Returns the last n formatted alerts.
        """
        return self.history[-n:]

    # ---------------------------------------------------------
    # Internal helpers
    # ---------------------------------------------------------
    def _store(self, message: str):
        """
        Stores a formatted alert message in the history buffer.
        """
        self.history.append(message)
        if len(self.history) > self.max_history:
            self.history.pop(0)
```

---

# 🧠 **Purpose**

`AlertManager` is the Analyzer’s **alert formatting and storage utility**.

It sits between:

- **AlertListener** (receives raw JSON alerts from the Generator)  
- **MainWindow / LogPanel / HealthSummary** (which need readable messages)

Its job is to:

- normalize alert dictionaries  
- format them into readable strings  
- store a rolling history  
- provide easy access to recent alerts  

This keeps the rest of the Analyzer clean and focused.

---

# 🧱 **Structure**

### 1. **History buffer**
Stores up to `max_history` alerts (default: 200).

### 2. **add_alert(alert_dict)**
- Extracts:
  - `event`
  - `timestamp`
  - `payload`
- Formats a readable message:
  ```
  [ALERT] chunk_written at 2026-02-17T10:42:00 — rows=10000
  ```
- Stores it in history  
- Returns the formatted string

### 3. **get_recent_alerts(n)**
Returns the last *n* formatted alerts.

---

# 📥 **Inputs**

From `AlertListener`:

```python
formatted = alert_manager.add_alert(alert_dict)
```

Where `alert_dict` looks like:

```json
{
  "event": "chunk_written",
  "timestamp": "2026-02-17T10:42:00",
  "payload": { "rows": 10000 }
}
```

---

# 📤 **Outputs**

### 1. **Formatted alert message**
Sent to:

- `LogPanel.append_log()`
- `HealthSummary.mark_data_updated()`
- internal debugging

### 2. **Stored history**
Used for:

- debugging  
- optional future GUI panels  
- exporting logs  

---

# 🔗 **Integration with the Analyzer**

### In `MainWindow.__init__`:

```python
self.alert_manager = AlertManager()
```

### In `_handle_alert()`:

```python
formatted = self.alert_manager.add_alert(alert_dict)
self.log_panel.append_log(formatted)

event = alert_dict.get("event")
if event == "chunk_written":
    self.health_summary.mark_data_updated()
elif event == "generation_complete":
    self.health_summary.mark_generation_complete()
```

---


## 5.2. file_monitor.py

Here is the **complete, production‑ready implementation** of `analyzer/utils/file_monitor.py`, written in the same clean, modular, real‑time‑friendly style as the rest of Project B.

This utility is intentionally simple and robust. It provides **fast, low‑overhead monitoring** of the telemetry file so the Analyzer can:

- detect file growth  
- detect resets or truncation  
- track total row count  
- support progress reporting  
- support alert‑driven refreshes  

It does **not** read the file itself — that’s the job of `TelemetryReader`.  
Instead, it tracks **metadata** only.

---

# 📄 `file_monitor.py`

```python
# analyzer/utils/file_monitor.py

import os
from typing import Optional
import pandas as pd


class FileMonitor:
    """
    Lightweight file monitor for telemetry files.

    Responsibilities:
        - Track file size changes
        - Track row count (CSV or Parquet)
        - Detect file resets or truncation
        - Provide metadata for progress reporting

    Methods:
        update() -> dict
            Returns a dictionary describing the current file state:
                {
                    "size_bytes": int,
                    "rows": int,
                    "reset_detected": bool
                }
    """

    def __init__(self, file_path: str):
        self.file_path = file_path

        # Internal state
        self.last_size: int = 0
        self.last_rows: int = 0

        # Detect format
        self.format = self._detect_format(file_path)

    # ---------------------------------------------------------
    # Format detection
    # ---------------------------------------------------------
    def _detect_format(self, path: str) -> str:
        ext = os.path.splitext(path)[1].lower()
        if ext == ".csv":
            return "csv"
        if ext == ".parquet":
            return "parquet"
        raise ValueError(f"Unsupported telemetry file format: {ext}")

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def update(self) -> dict:
        """
        Checks the current file state and returns metadata.

        Returns:
            {
                "size_bytes": int,
                "rows": int,
                "reset_detected": bool
            }
        """
        if not os.path.exists(self.file_path):
            return {
                "size_bytes": 0,
                "rows": 0,
                "reset_detected": False,
            }

        size = os.path.getsize(self.file_path)
        rows = self._count_rows()

        # Detect file reset (e.g., generator restarted)
        reset = size < self.last_size or rows < self.last_rows

        # Update internal state
        self.last_size = size
        self.last_rows = rows

        return {
            "size_bytes": size,
            "rows": rows,
            "reset_detected": reset,
        }

    # ---------------------------------------------------------
    # Row counting
    # ---------------------------------------------------------
    def _count_rows(self) -> int:
        """
        Efficient row counting for CSV and Parquet.
        """
        try:
            if self.format == "csv":
                return self._count_csv_rows()
            else:
                return self._count_parquet_rows()
        except Exception:
            return 0

    def _count_csv_rows(self) -> int:
        """
        Counts rows in a CSV file without loading the full file.
        """
        with open(self.file_path, "r", encoding="utf-8") as f:
            # subtract header
            return max(0, sum(1 for _ in f) - 1)

    def _count_parquet_rows(self) -> int:
        """
        Counts rows in a Parquet file using metadata only.
        """
        meta = pd.read_parquet(self.file_path, columns=[])
        return meta.shape[0]
```

---

# 🧠 **Purpose**

`FileMonitor` is the Analyzer’s **metadata watcher**.  
It provides fast, low‑overhead tracking of:

- file size  
- row count  
- file resets (e.g., Generator restarted)  

This is essential for:

- progress reporting  
- health monitoring  
- detecting when the Generator starts over  
- synchronizing with alert events  

It never loads the full file — it only inspects metadata.

---

# 🧱 **Structure**

### 1. **Format detection**
Automatically identifies CSV or Parquet.

### 2. **update()**
Returns a dictionary:

```python
{
  "size_bytes": 123456,
  "rows": 100000,
  "reset_detected": False
}
```

### 3. **Row counting**
- CSV: counts lines without loading data  
- Parquet: uses metadata only  

### 4. **Reset detection**
If file size or row count decreases → file was reset.

---

# 📥 **Inputs**

From `AnalyzerLoop` or `MainWindow`:

```python
state = file_monitor.update()
```

Where `file_path` is the telemetry file.

---

# 📤 **Outputs**

A dictionary describing the current file state:

```python
{
  "size_bytes": int,
  "rows": int,
  "reset_detected": bool
}
```

This can be used for:

- progress bar updates  
- log messages  
- health warnings  
- detecting generator restarts  

---

# 🔗 **Integration with AnalyzerLoop**

Inside `_run_loop()` (optional enhancement):

```python
file_state = self.file_monitor.update()
if file_state["reset_detected"]:
    self.log_callback("Telemetry file reset detected.")
    self.reader.last_row_index = 0
```

---


## 5.3. 

Below is the **complete, production‑ready implementation** of `analyzer/utils/plot_helpers.py`, written to match the architecture and design philosophy of Project B: clean, modular, lightweight, and GUI‑friendly.

This utility provides **small, reusable helpers** for converting raw module outputs into formats that the GUI’s `VisualizationTabs` can easily consume.  
It does **not** perform any plotting itself — Qt handles that — but it standardizes and sanitizes the data.

---

# 📄 `plot_helpers.py`

```python
# analyzer/utils/plot_helpers.py

import numpy as np
from typing import Dict, Any, List, Optional


class PlotHelpers:
    """
    Utility class for preparing visualization-friendly data structures.

    Responsibilities:
        - Normalize time-series data
        - Validate scatterplot inputs
        - Prepare anomaly score curves
        - Provide safe defaults for missing or malformed data

    Methods:
        prepare_time_series(result_dict)
        prepare_scatter(result_dict)
        prepare_forecast(result_dict)
        prepare_anomaly_curve(result_dict)
    """

    # ---------------------------------------------------------
    # Time-Series
    # ---------------------------------------------------------
    @staticmethod
    def prepare_time_series(data: Dict[str, Any]) -> Dict[str, List[float]]:
        """
        Ensures time-series data is well-formed.

        Expected:
            {
                "x": [...],
                "y": [...],
                "label": "Temperature"
            }
        """
        if not data:
            return {"x": [], "y": [], "label": "No Data"}

        x = data.get("x", [])
        y = data.get("y", [])
        label = data.get("label", "Signal")

        # Convert to lists of floats
        try:
            x = [float(v) for v in x]
            y = [float(v) for v in y]
        except Exception:
            x, y = [], []

        return {"x": x, "y": y, "label": label}

    # ---------------------------------------------------------
    # Scatter (Clustering)
    # ---------------------------------------------------------
    @staticmethod
    def prepare_scatter(data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Ensures clustering scatterplot data is well-formed.

        Expected:
            {
                "x": [...],
                "y": [...],
                "labels": [...],
                "columns": [...],
                "method": "kmeans"
            }
        """
        if not data:
            return {"x": [], "y": [], "labels": [], "columns": [], "method": "none"}

        try:
            x = [float(v) for v in data.get("x", [])]
            y = [float(v) for v in data.get("y", [])]
            labels = data.get("labels", [])
        except Exception:
            x, y, labels = [], [], []

        return {
            "x": x,
            "y": y,
            "labels": labels,
            "columns": data.get("columns", []),
            "method": data.get("method", "unknown"),
        }

    # ---------------------------------------------------------
    # Forecasting
    # ---------------------------------------------------------
    @staticmethod
    def prepare_forecast(data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Ensures forecast data is well-formed.

        Expected:
            {
                "history_x": [...],
                "history_y": [...],
                "forecast_x": [...],
                "forecast_y": [...],
                "label": "Temperature"
            }
        """
        if not data:
            return {
                "history_x": [],
                "history_y": [],
                "forecast_x": [],
                "forecast_y": [],
                "label": "No Data",
            }

        try:
            hx = [float(v) for v in data.get("history_x", [])]
            hy = [float(v) for v in data.get("history_y", [])]
            fx = [float(v) for v in data.get("forecast_x", [])]
            fy = [float(v) for v in data.get("forecast_y", [])]
        except Exception:
            hx, hy, fx, fy = [], [], [], []

        return {
            "history_x": hx,
            "history_y": hy,
            "forecast_x": fx,
            "forecast_y": fy,
            "label": data.get("label", "Signal"),
        }

    # ---------------------------------------------------------
    # Anomaly Curve (Deep Learning)
    # ---------------------------------------------------------
    @staticmethod
    def prepare_anomaly_curve(data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Ensures anomaly score data is well-formed.

        Expected:
            {
                "x": [...],
                "anomaly_score": [...],
                "columns": [...]
            }
        """
        if not data:
            return {"x": [], "anomaly_score": [], "columns": []}

        try:
            x = [float(v) for v in data.get("x", [])]
            scores = [float(v) for v in data.get("anomaly_score", [])]
        except Exception:
            x, scores = [], []

        return {
            "x": x,
            "anomaly_score": scores,
            "columns": data.get("columns", []),
        }
```

---

# 🧠 **Purpose**

`plot_helpers.py` is the Analyzer’s **data‑sanitization layer** for visualization.

It ensures that every module’s output:

- is numeric  
- is well‑formed  
- is safe to pass to Qt plotting widgets  
- has reasonable defaults if something goes wrong  

This prevents GUI crashes and keeps the visualization layer clean and predictable.

---

# 📥 **Inputs**

Each method receives the raw output from a module, e.g.:

- `statistics.run()`  
- `clustering.run()`  
- `forecasting.run()`  
- `deep_learning.run()`  

Example:

```python
prepared = PlotHelpers.prepare_time_series(stats_result)
```

---

# 📤 **Outputs**

Each method returns a **cleaned, validated dictionary** ready for plotting.

Examples:

### Time series
```python
{"x": [...], "y": [...], "label": "Temperature"}
```

### Clustering
```python
{"x": [...], "y": [...], "labels": [...], "columns": [...], "method": "kmeans"}
```

### Forecasting
```python
{"history_x": [...], "history_y": [...], "forecast_x": [...], "forecast_y": [...], "label": "Temperature"}
```

### Anomaly curve
```python
{"x": [...], "anomaly_score": [...], "columns": [...]}
```

---

# 🔗 **Integration with the Analyzer**

Inside `VisualizationTabs.update_visualizations()`:

```python
if "time_series" in results:
    ts = PlotHelpers.prepare_time_series(results["time_series"])
    self.update_time_series(ts)

if "clustering" in results:
    sc = PlotHelpers.prepare_scatter(results["clustering"])
    self.update_clustering(sc)

if "forecasting" in results:
    fc = PlotHelpers.prepare_forecast(results["forecasting"])
    self.update_forecasting(fc)

if "deep_learning" in results:
    dl = PlotHelpers.prepare_anomaly_curve(results["deep_learning"])
    self.update_deep_learning(dl)
```

---


# 6. Analyzer Folder

## 6.1. app.py


This is the **clean, minimal entry point** that wires the whole Analyzer together.

---

### 📄 `app.py`

```python
# analyzer/app.py

import sys
import os

from PySide6.QtWidgets import QApplication
from PySide6.QtCore import Qt

from .gui.main_window import MainWindow


def main():
    """
    Entry point for the Telemetry Analyzer (Project B).

    Responsibilities:
        - Initialize Qt application
        - Create and show MainWindow
        - Optionally accept a config path via CLI
    """
    app = QApplication(sys.argv)
    app.setApplicationName("Telemetry Analyzer")
    app.setAttribute(Qt.AA_EnableHighDpiScaling, True)

    # Optional: config path from CLI, default: ./config.json
    if len(sys.argv) > 1:
        config_path = sys.argv[1]
    else:
        config_path = os.path.join(os.getcwd(), "config.json")

    window = MainWindow(config_path=config_path)
    window.show()

    sys.exit(app.exec())


if __name__ == "__main__":
    main()
```

---

### 🧠 Purpose

- Provides a **single, explicit entry point** for Project B.  
- Initializes the Qt event loop and shows `MainWindow`.  
- Accepts an optional `config.json` path as a command‑line argument, otherwise uses the current working directory.

---

### 📥 Inputs

- Optional CLI argument: `python -m analyzer.app path/to/config.json`  
- If omitted, it uses `./config.json`.

---

### 📤 Outputs

- Launches the full Analyzer GUI:
  - loads config via `MainWindow`  
  - starts `AlertListener`  
  - prepares `AnalyzerLoop`, `HealthSummary`, `VisualizationTabs`, etc.

---


# 7. Start the application

## 7.1. Install python packages

Here is a **clean, unified pip‑install list** we can paste directly into a Jupyter Notebook cell to install *all* required packages for **both Project A (Generator)** and **Project B (Analyzer)**.

This list is fully consistent with the lightweight architecture we built:  
no NLTK, no spaCy, no Torch, no SHAP — everything runs fast and clean.

---

# ✅ **Pip install list for Jupyter Notebook (Projects A + B)**

We can paste this directly into a Jupyter cell:

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

This installs:

| Package | Purpose |
|--------|---------|
| **numpy** | Core numerical operations |
| **pandas** | DataFrames, CSV/Parquet handling |
| **scikit-learn** | Clustering, forecasting, anomaly detection, XAI |
| **pyarrow** | Parquet read/write for Generator + Analyzer |
| **PySide6** | GUI framework for Analyzer |
| **matplotlib** | Plotting inside the Analyzer |
| **scipy** | Optional FFT and numerical utilities |
| **tqdm** | Progress bars for Generator |

---

# 🧠 Why this list is complete

Because of our design choices:

- **NLP** uses regex + Counter → no NLTK/spaCy  
- **Deep Learning** uses IsolationForest → no PyTorch/TensorFlow  
- **XAI** uses RandomForestRegressor → no SHAP  
- **Forecasting** uses LinearRegression → no statsmodels/Prophet  

This keeps the environment:

- lightweight  
- fast  
- easy to install  
- fully compatible with Jupyter  

---


!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm

## 7.2. Run instructions

Let’s make this really concrete and reproducible: we are in a Jupyter Notebook, and we want to run **Project A (Generator)** and **Project B (Analyzer)** from there, using the codebase we’ve built.

I’ll assume a layout like:

```text
project-root/
  generator/
    app.py
    ...
  analyzer/
    app.py
    ...
  config.json
```

We’re running Jupyter with the working directory set to `project-root`.

---

## 1. Install all dependencies from the notebook

In a fresh notebook cell:

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart the kernel after installation if needed.

---

## 2. Make sure Python can see your packages

If `generator` and `analyzer` are plain folders (no installation as packages), add the project root to `sys.path`:

```python
import sys, os

project_root = os.path.abspath(".")  # or explicit path
if project_root not in sys.path:
    sys.path.append(project_root)
```

Now `import generator` and `import analyzer` will work.

---

## 3. Running Project A (Generator) from Jupyter

You have two main options:

### 3.1. Run as a script via `!python`

If `generator/app.py` is the entry point:

```python
!python -m generator.app
```

If `app.py` expects arguments (e.g., output file, config path), you can pass them:

```python
!python -m generator.app --config config.json
```

This runs the Generator as a normal script from inside the notebook.  
It will:

- read/write `config.json`  
- generate `telemetry.parquet` (or CSV)  
- possibly run until completion or in a loop, depending on your implementation.

### 3.2. Run by importing and calling `main()`

If `generator/app.py` exposes a `main()` function:

```python
from generator.app import main as generator_main

generator_main()
```

This keeps everything in the same Python process.  
If the Generator is long‑running (e.g., infinite stream), you might want to run it in a background thread or a separate terminal instead—Jupyter will block until it finishes.

---

## 4. Running Project B (Analyzer) from Jupyter

The Analyzer is a **Qt GUI application** (PySide6). That means:

- it starts its own event loop (`app.exec()`)  
- it opens a separate window  
- it should ideally run in its own process, not inside the Jupyter kernel’s event loop

So the cleanest way from Jupyter is to call it as a script.

### 4.1. Run Analyzer as a script via `!python`

Assuming `analyzer/app.py` is the entry point we defined:

```python
!python -m analyzer.app
```

If you want to pass a specific `config.json`:

```python
!python -m analyzer.app config.json
```

This will:

- start a separate process  
- open the Analyzer GUI window  
- read `config.json`  
- start `AlertListener`, `AnalyzerLoop`, etc.

Your Jupyter kernel remains free.

> This is the most robust way: Jupyter for orchestration, Analyzer in its own process.

### 4.2. Import and call `main()` (possible, but less ideal)

You *can* do:

```python
from analyzer.app import main as analyzer_main

analyzer_main()
```

But:

- the Qt event loop will block the notebook cell until the GUI is closed  
- mixing Qt event loops and Jupyter’s event loop can be fragile  

I’d only use this for quick tests, not for regular work.

---

## 5. End‑to‑end workflow from a single notebook

Here’s a realistic, reproducible flow.

### 5.1. Setup cell

```python
import sys, os

project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.append(project_root)

!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart kernel if needed, then re‑run the `sys.path` cell.

### 5.2. Generate telemetry with Project A

Option A — run once, generate a finite file:

```python
!python -m generator.app
```

Option B — if you have a `main()`:

```python
from generator.app import main as generator_main

generator_main()  # blocks until done
```

After this, you should have:

- `config.json`  
- `telemetry.parquet` (or CSV)  

in your project root (or configured paths).

### 5.3. Start Analyzer (Project B) to inspect the data

In a new cell:

```python
!python -m analyzer.app config.json
```

This opens the GUI:

- reads `config.json`  
- loads `telemetry.parquet`  
- runs all selected modules (statistics, clustering, forecasting, NLP, deep_learning, xai)  
- updates plots and health indicators  

You can keep the Analyzer running while you:

- regenerate data with the Generator  
- tweak parameters in the notebook  
- re‑run the Generator to append more telemetry  

If your Generator sends socket alerts, the Analyzer will react in real time.

---

## 6. Advanced: orchestrating both from Jupyter

If you want to orchestrate both from one notebook, you can:

- run the Analyzer in a separate process  
- run the Generator in another cell (or another process)  
- use `config.json` + socket alerts as the contract

Example:

```python
# Cell 1: start Analyzer (non-blocking, in background shell)
!python -m analyzer.app config.json &
```

```python
# Cell 2: run Generator
!python -m generator.app
```

(Backgrounding with `&` works in some Jupyter setups; in others you might prefer a separate terminal.)

---


With our folder structure:

```
D:\DigitalTwinsGeneratorGUI\
    analyzer\
        app.py
        ...
    generator\
        app.py
        ...
    config.json
```

we can run **both Project A and Project B directly from a single Jupyter Notebook** using a clean, reproducible runner notebook.

Below is a complete, ready‑to‑use **Jupyter Notebook runner** that:

- sets up the environment  
- adds our project to `sys.path`  
- runs **Generator** (Project A)  
- runs **Analyzer** (Project B GUI)  
- supports background execution  
- works on Windows (your setup)  

---

# 📘 **Jupyter Notebook Runner for Project A + B**

Create a new notebook in:

```
D:\DigitalTwinsGeneratorGUI\run.ipynb
```

and paste the following cells.

---

# 🟦 **Cell 1 — Setup environment & paths**

```python
import sys
import os

# Path to your main folder
PROJECT_ROOT = r"D:\DigitalTwinsGeneratorGUI"

# Add to sys.path so Python can import generator and analyzer
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)
```

This ensures:

```python
from generator.app import main
from analyzer.app import main
```

works correctly.

---

# 🟦 **Cell 2 — Install all required packages**

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart the kernel once after installation.

---

# 🟦 **Cell 3 — Run Project A (Generator)**

You have two clean options.

---

## **Option A — Run Generator as a separate process (recommended)**  
This keeps Jupyter responsive.

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

If your generator accepts a config path:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

---

## **Option B — Run Generator inside the notebook (blocks until done)**

```python
from generator.app import main as generator_main

generator_main()
```

Use this only if the generator finishes quickly.  
If it streams indefinitely, prefer Option A.

---

# 🟦 **Cell 4 — Run Project B (Analyzer GUI)**

The Analyzer is a **PySide6 Qt GUI**, so it must run in its own process.  
Running it inside the notebook would block the kernel and conflict with Jupyter’s event loop.

So we run it as a separate process:

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

This will:

- open the Analyzer GUI window  
- load `config.json`  
- start the alert listener  
- start the analysis loop  
- update plots in real time  

Our Jupyter Notebook remains free to run other cells.

---

# 🟦 **Cell 5 — (Optional) Run Analyzer in background**

If you want the GUI to start but keep the notebook cell free immediately:

```python
import subprocess

analyzer_process = subprocess.Popen(
    ["python", r"D:\DigitalTwinsGeneratorGUI\analyzer\app.py", r"D:\DigitalTwinsGeneratorGUI\config.json"]
)

print("Analyzer started with PID:", analyzer_process.pid)
```

You can later stop it:

```python
analyzer_process.terminate()
```

---

# 🟦 **Cell 6 — (Optional) Regenerate telemetry while Analyzer is running**

If the Analyzer is open and listening for alerts, you can regenerate data:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

The Analyzer will:

- detect file changes  
- receive socket alerts  
- refresh plots  
- update health indicators  

This gives us a full **real‑time digital twin loop** from inside Jupyter.

---

# 🎯 **Summary: How to run both projects from Jupyter**

| Task | Best Method | Why |
|------|-------------|------|
| Run Generator | `!python generator/app.py` | Non‑blocking, clean |
| Run Analyzer | `!python analyzer/app.py` | Required for Qt GUI |
| Run both simultaneously | Analyzer in background + Generator in foreground | Real‑time loop |
| Import and run inside notebook | Only for Generator | Analyzer must run in separate process |

---
